In [1]:
# =============================================================================
# TIF STUDY — CELL 0
# SINGLE-PASS SYNTHETIC TEMPORAL NETWORK + DENSE OBSERVABLE TRAJECTORY
#
# Method: Temporal Interaction Factorization (TIF)
#
# Purpose
# -------
# 1. Define the N=8 synthetic ground-truth system.
# 2. Activate G1 -> G2 -> ... -> G6 exactly once.
# 3. Generate one densely observed continuous-state trajectory.
# 4. Strictly separate observable data from synthetic oracle information.
#
# IMPORTANT
# ---------
# All later inference cells should use ONLY:
#
#     TIF_DATA_T
#     TIF_DATA_X
#     TIF_DT
#
# They must NOT use TIF_ORACLE until the final validation stage.
# =============================================================================

import numpy as np
from itertools import combinations


# =============================================================================
# 0. GLOBAL EXPERIMENT SETTINGS
# =============================================================================

N = 8
TIF_SEED = 20260811

TIF_DT = 5.0e-4          # dense observation interval
TIF_STAGE_DURATION = 0.060
TIF_N_STAGES = 6

TIF_STEPS_PER_STAGE = int(round(TIF_STAGE_DURATION / TIF_DT))
TIF_TOTAL_STEPS = TIF_N_STAGES * TIF_STEPS_PER_STAGE
TIF_TOTAL_TIME = TIF_N_STAGES * TIF_STAGE_DURATION

assert np.isclose(
    TIF_STEPS_PER_STAGE * TIF_DT,
    TIF_STAGE_DURATION
)

print("=" * 92)
print("TEMPORAL INTERACTION FACTORIZATION (TIF) — CELL 0")
print("SINGLE-PASS DENSE TEMPORAL TRAJECTORY")
print("=" * 92)

print("\nExperiment geometry")
print("-" * 92)
print(f"N                           : {N}")
print(f"number of temporal stages   : {TIF_N_STAGES}")
print(f"duration per stage          : {TIF_STAGE_DURATION:.6f}")
print(f"observation interval dt     : {TIF_DT:.6f}")
print(f"steps per stage             : {TIF_STEPS_PER_STAGE}")
print(f"total integration steps     : {TIF_TOTAL_STEPS}")
print(f"total trajectory duration   : {TIF_TOTAL_TIME:.6f}")


# =============================================================================
# 1. SYNTHETIC ORACLE — POSSIBLE PAIR COUPLINGS
#
# 1-indexed labels are retained here for readability.
# These variables are ground truth and must NOT enter blind inference.
# =============================================================================

TRUE_EDGE_WEIGHTS_1B = {
    (1, 2): 1.006,
    (2, 3): 0.911,
    (3, 4): 1.105,
    (4, 5): 0.917,
    (5, 6): 1.177,
    (6, 7): 1.119,
    (7, 8): 0.917,
    (8, 1): 0.944,
    (1, 3): 1.026,
    (2, 4): 1.002,
    (3, 5): 1.067,
    (5, 7): 1.031,
}

TRUE_STAGE_EDGES_1B = (
    ((1, 2), (3, 4), (5, 6), (7, 8)),   # G1
    ((2, 3), (4, 5), (6, 7), (8, 1)),   # G2
    ((1, 3), (2, 4), (3, 5), (5, 7)),   # G3
    ((1, 2), (4, 5), (7, 8), (3, 5)),   # G4
    ((2, 3), (5, 6), (8, 1), (5, 7)),   # G5
    ((3, 4), (6, 7), (1, 3), (2, 4)),   # G6
)


# =============================================================================
# 2. SYNTHETIC ORACLE — PERSISTENT TRIADS
# =============================================================================

TRUE_TRIADS_1B = (
    (1, 2, 3),
    (2, 5, 8),
)

TRUE_TRIAD_STRENGTH = 0.020


# =============================================================================
# 3. INTERNAL ZERO-BASED ORACLE REPRESENTATION
# =============================================================================

def canonical_pair_0b(i, j):
    """Canonical zero-based undirected pair."""
    return tuple(sorted((i - 1, j - 1)))


TRUE_EDGE_WEIGHTS = {
    canonical_pair_0b(i, j): w
    for (i, j), w in TRUE_EDGE_WEIGHTS_1B.items()
}

TRUE_STAGE_EDGES = tuple(
    tuple(canonical_pair_0b(i, j) for i, j in stage)
    for stage in TRUE_STAGE_EDGES_1B
)

TRUE_TRIADS = tuple(
    tuple(v - 1 for v in triad)
    for triad in TRUE_TRIADS_1B
)


# =============================================================================
# 4. TRUE INTERACTION LAWS
#
# These are used ONLY to generate synthetic data.
# TIF inference will later pretend that these functional forms are unknown.
#
# Pair law:
#
#       phi(x) = x + 1/2 x^2
#
# Pair field on edge (i,j):
#
#       F_i = w_ij [phi(x_j) - phi(x_i)]
#       F_j = -F_i
#
# Persistent triad law:
#
#       T_i = x_j x_k - 1/2 x_i x_j - 1/2 x_i x_k
#
# with cyclic permutations for j and k.
# =============================================================================

def true_phi(x):
    return x + 0.5 * x**2


def true_pair_contribution(x, i, j, weight):
    """
    Conservative pair contribution for one active edge.
    """
    out = np.zeros(N, dtype=float)

    flux = weight * (true_phi(x[j]) - true_phi(x[i]))

    out[i] += flux
    out[j] -= flux

    return out


def true_triad_contribution(x, i, j, k, strength):
    """
    Conservative permutation-equivariant triadic contribution.
    """
    out = np.zeros(N, dtype=float)

    xi, xj, xk = x[i], x[j], x[k]

    Ti = (
        xj * xk
        - 0.5 * xi * xj
        - 0.5 * xi * xk
    )

    Tj = (
        xk * xi
        - 0.5 * xj * xk
        - 0.5 * xj * xi
    )

    Tk = (
        xi * xj
        - 0.5 * xk * xi
        - 0.5 * xk * xj
    )

    out[i] += strength * Ti
    out[j] += strength * Tj
    out[k] += strength * Tk

    return out


def true_rhs(x, stage_index):
    """
    Ground-truth instantaneous vector field for one temporal stage.

    stage_index = 0,...,5 corresponds to G1,...,G6.
    """
    dx = np.zeros(N, dtype=float)

    # Active pair interactions
    for i, j in TRUE_STAGE_EDGES[stage_index]:
        w = TRUE_EDGE_WEIGHTS[(i, j)]
        dx += true_pair_contribution(x, i, j, w)

    # Persistent native triadic interactions
    for i, j, k in TRUE_TRIADS:
        dx += true_triad_contribution(
            x,
            i,
            j,
            k,
            TRUE_TRIAD_STRENGTH,
        )

    return dx


# =============================================================================
# 5. CONSERVATION AUDIT OF THE TRUE VECTOR FIELD
# =============================================================================

rng_audit = np.random.default_rng(TIF_SEED + 1)

max_rhs_conservation_error = 0.0

for _ in range(64):
    x_probe = rng_audit.uniform(-0.8, 0.8, size=N)

    for stage_index in range(TIF_N_STAGES):
        err = abs(np.sum(true_rhs(x_probe, stage_index)))
        max_rhs_conservation_error = max(
            max_rhs_conservation_error,
            err,
        )

print("\nGround-truth field audit")
print("-" * 92)
print(
    "max |sum_i F_i|             : "
    f"{max_rhs_conservation_error:.3e}"
)

assert max_rhs_conservation_error < 1e-12


# =============================================================================
# 6. INITIAL CONDITION
#
# Fresh deterministic initial condition for the new TIF notebook.
#
# We deliberately use a heterogeneous, non-symmetric state to provide
# trajectory excitation. The mean is removed so that the conserved total
# state is exactly zero up to floating-point precision.
# =============================================================================

rng_x0 = np.random.default_rng(TIF_SEED)

TIF_X0 = rng_x0.uniform(
    low=-0.65,
    high=0.65,
    size=N,
)

TIF_X0 -= np.mean(TIF_X0)

print("\nInitial condition")
print("-" * 92)
print("x0                          :", np.round(TIF_X0, 6))
print(f"sum(x0)                     : {np.sum(TIF_X0):.3e}")
print(f"||x0||_2                    : {np.linalg.norm(TIF_X0):.6f}")


# =============================================================================
# 7. FIXED-STEP RK4
#
# Stage boundaries fall exactly on observation-grid points:
#
#     0.06 / 0.0005 = 120
#
# Therefore no integration interval straddles a topology switch.
# =============================================================================

def rk4_step(x, dt, rhs):
    k1 = rhs(x)
    k2 = rhs(x + 0.5 * dt * k1)
    k3 = rhs(x + 0.5 * dt * k2)
    k4 = rhs(x + dt * k3)

    return x + (dt / 6.0) * (
        k1 + 2.0 * k2 + 2.0 * k3 + k4
    )


# =============================================================================
# 8. GENERATE SINGLE-PASS TRAJECTORY
#
# G1 -> G2 -> G3 -> G4 -> G5 -> G6
#
# No repetition.
# =============================================================================

TIF_DATA_T = np.arange(
    TIF_TOTAL_STEPS + 1,
    dtype=float,
) * TIF_DT

TIF_DATA_X = np.empty(
    (TIF_TOTAL_STEPS + 1, N),
    dtype=float,
)

TIF_DATA_X[0] = TIF_X0.copy()

x = TIF_X0.copy()
cursor = 0

for stage_index in range(TIF_N_STAGES):

    rhs_stage = lambda state, s=stage_index: true_rhs(state, s)

    for _ in range(TIF_STEPS_PER_STAGE):
        x = rk4_step(
            x,
            TIF_DT,
            rhs_stage,
        )

        cursor += 1
        TIF_DATA_X[cursor] = x


assert cursor == TIF_TOTAL_STEPS


# =============================================================================
# 9. OBSERVABLE DATA OBJECT
#
# This is the ONLY object family later inference cells are allowed to use.
#
# No stage labels.
# No topology.
# No true weights.
# No true interaction-law coefficients.
# =============================================================================

TIF_DATA = {
    "t": TIF_DATA_T,
    "x": TIF_DATA_X,
    "dt": TIF_DT,
}


# =============================================================================
# 10. SYNTHETIC ORACLE OBJECT
#
# Keep this closed until final validation.
# =============================================================================

TIF_ORACLE = {
    "stage_duration": TIF_STAGE_DURATION,
    "stage_edges_1b": TRUE_STAGE_EDGES_1B,
    "edge_weights_1b": TRUE_EDGE_WEIGHTS_1B,
    "triads_1b": TRUE_TRIADS_1B,
    "triad_strength": TRUE_TRIAD_STRENGTH,
    "pair_law_coefficients": np.array([1.0, 0.5]),
    "triad_law_coefficients": np.array([0.0, 0.0, 1.0]),
}


# =============================================================================
# 11. TRAJECTORY INTEGRITY CHECKS
# =============================================================================

trajectory_sum = np.sum(TIF_DATA_X, axis=1)

max_conservation_drift = np.max(
    np.abs(trajectory_sum - trajectory_sum[0])
)

all_finite = np.all(np.isfinite(TIF_DATA_X))

trajectory_displacement = np.linalg.norm(
    TIF_DATA_X[-1] - TIF_DATA_X[0]
)

state_min = np.min(TIF_DATA_X)
state_max = np.max(TIF_DATA_X)


print("\nObservable trajectory")
print("-" * 92)

print(f"time samples                  : {len(TIF_DATA_T)}")
print(f"state-array shape             : {TIF_DATA_X.shape}")
print(f"time range                    : "
      f"[{TIF_DATA_T[0]:.6f}, {TIF_DATA_T[-1]:.6f}]")

print(f"state range                   : "
      f"[{state_min:.6f}, {state_max:.6f}]")

print(f"trajectory displacement       : "
      f"{trajectory_displacement:.6f}")

print(f"max conservation drift        : "
      f"{max_conservation_drift:.3e}")

print(f"all states finite             : {all_finite}")


assert TIF_DATA_X.shape == (721, N)
assert len(TIF_DATA_T) == 721
assert all_finite
assert max_conservation_drift < 1e-11


# =============================================================================
# 12. BLIND-INFERENCE CONTRACT
# =============================================================================

print("\nBlind-inference contract")
print("-" * 92)

print("observable input available    : t, x(t), dt")
print("number of stages revealed     : NO")
print("switching times revealed      : NO")
print("active topology revealed      : NO")
print("interaction strengths revealed: NO")
print("pair interaction law revealed : NO")
print("triad interaction law revealed: NO")
print("repeated cycles used          : NO")

print("\n" + "=" * 92)
print("CELL 0 PASSED")
print("=" * 92)

TEMPORAL INTERACTION FACTORIZATION (TIF) — CELL 0
SINGLE-PASS DENSE TEMPORAL TRAJECTORY

Experiment geometry
--------------------------------------------------------------------------------------------
N                           : 8
number of temporal stages   : 6
duration per stage          : 0.060000
observation interval dt     : 0.000500
steps per stage             : 120
total integration steps     : 720
total trajectory duration   : 0.360000

Ground-truth field audit
--------------------------------------------------------------------------------------------
max |sum_i F_i|             : 6.661e-16

Initial condition
--------------------------------------------------------------------------------------------
x0                          : [-0.319989 -0.301418  0.354093 -0.213312 -0.026615  0.067347  0.282008
  0.157887]
sum(x0)                     : -1.110e-16
||x0||_2                    : 0.688353

Observable trajectory
--------------------------------------------------------------

In [2]:
# =============================================================================
# TIF STUDY — CELL 1
# OBSERVABLE GRADIENT MATCHING + BLIND TEMPORAL CHANGE-POINT DETECTION
#
# Input
# -----
# ONLY:
#
#     TIF_DATA["t"]
#     TIF_DATA["x"]
#     TIF_DATA["dt"]
#
# No oracle topology.
# No stage count.
# No switching times.
# No interaction law.
#
# Purpose
# -------
# A. Construct midpoint gradient-matching observations:
#
#       xbar_n = (x_n + x_{n+1}) / 2
#
#       v_n    = (x_{n+1} - x_n) / dt
#
#    so that, away from temporal switches,
#
#       v_n = F(xbar_n, t_{n+1/2}) + O(dt^2).
#
# B. Detect temporal regime changes directly from jumps in the observable
#    velocity sequence.
#
# C. Audit the numerical derivative using a same-midpoint 3dt secant,
#    without access to the true vector field.
# =============================================================================

import numpy as np


# =============================================================================
# 0. READ OBSERVABLE DATA ONLY
# =============================================================================

t = np.asarray(TIF_DATA["t"], dtype=float)
X = np.asarray(TIF_DATA["x"], dtype=float)
dt = float(TIF_DATA["dt"])

n_states, n_nodes = X.shape
n_intervals = n_states - 1

assert len(t) == n_states
assert n_nodes == N
assert np.allclose(np.diff(t), dt)


print("=" * 92)
print("TEMPORAL INTERACTION FACTORIZATION (TIF) — CELL 1")
print("OBSERVABLE GRADIENT MATCHING + BLIND CHANGE-POINT DETECTION")
print("=" * 92)

print("\nObservable input")
print("-" * 92)
print(f"state samples                 : {n_states}")
print(f"transition intervals          : {n_intervals}")
print(f"nodes                         : {n_nodes}")
print(f"dt                            : {dt:.6e}")


# =============================================================================
# 1. MIDPOINT GRADIENT-MATCHING OBSERVABLES
#
# Interval n:
#
#       [t_n, t_{n+1}]
#
# Observable midpoint:
#
#       xbar_n = (x_n + x_{n+1}) / 2
#
# Observable secant velocity:
#
#       v_n = (x_{n+1} - x_n) / dt
#
# For a smooth autonomous field within the interval:
#
#       v_n = F(xbar_n) + O(dt^2).
# =============================================================================

TIF_GM_T = 0.5 * (t[:-1] + t[1:])

TIF_GM_X = 0.5 * (
    X[:-1] + X[1:]
)

TIF_GM_V = (
    X[1:] - X[:-1]
) / dt


assert TIF_GM_T.shape == (n_intervals,)
assert TIF_GM_X.shape == (n_intervals, N)
assert TIF_GM_V.shape == (n_intervals, N)


# =============================================================================
# 2. BASIC OBSERVABLE VELOCITY AUDIT
# =============================================================================

velocity_norm = np.linalg.norm(
    TIF_GM_V,
    axis=1,
)

velocity_conservation_error = np.abs(
    np.sum(TIF_GM_V, axis=1)
)

print("\nMidpoint gradient-matching observables")
print("-" * 92)

print(f"midpoint states shape         : {TIF_GM_X.shape}")
print(f"velocity array shape          : {TIF_GM_V.shape}")

print(
    f"velocity norm range           : "
    f"[{np.min(velocity_norm):.6e}, "
    f"{np.max(velocity_norm):.6e}]"
)

print(
    f"median velocity norm          : "
    f"{np.median(velocity_norm):.6e}"
)

print(
    f"max |sum_i v_i|               : "
    f"{np.max(velocity_conservation_error):.3e}"
)


# =============================================================================
# 3. PURELY OBSERVABLE TEMPORAL-JUMP SIGNAL
#
# Consecutive interval velocities:
#
#       v_n
#       v_{n+1}
#
# Their difference is:
#
#       O(dt)
#
# inside a smooth temporal regime, but becomes O(1) if the vector field
# changes discontinuously at the common state time t_{n+1}.
#
# Define:
#
#       J_n = ||v_{n+1} - v_n||_2
#
# The associated candidate change time is:
#
#       t_{n+1}.
# =============================================================================

TIF_JUMP_TIME = t[1:-1]

TIF_JUMP_ABS = np.linalg.norm(
    TIF_GM_V[1:] - TIF_GM_V[:-1],
    axis=1,
)

local_velocity_scale = 0.5 * (
    velocity_norm[1:] + velocity_norm[:-1]
)

TIF_JUMP_REL = (
    TIF_JUMP_ABS
    /
    np.maximum(local_velocity_scale, 1.0e-14)
)


# =============================================================================
# 4. ROBUST BLIND THRESHOLD
#
# We do NOT specify:
#
#       - number of stages
#       - number of change points
#       - stage duration
#
# Instead use a conservative robust outlier criterion:
#
#       median(J) + 20 * robust_sigma
#
# with
#
#       robust_sigma = 1.4826 * MAD.
#
# In the noiseless synthetic proof-of-concept, genuine switching events
# should be separated from smooth within-regime curvature by orders of
# magnitude.
# =============================================================================

jump_median = np.median(TIF_JUMP_ABS)

jump_mad = np.median(
    np.abs(TIF_JUMP_ABS - jump_median)
)

jump_robust_sigma = 1.4826 * jump_mad

TIF_JUMP_THRESHOLD = (
    jump_median
    +
    20.0 * max(
        jump_robust_sigma,
        np.finfo(float).eps,
    )
)

TIF_CP_JUMP_INDEX = np.flatnonzero(
    TIF_JUMP_ABS > TIF_JUMP_THRESHOLD
)

TIF_CP_TIME = TIF_JUMP_TIME[
    TIF_CP_JUMP_INDEX
]


# =============================================================================
# 5. REPORT STRONGEST OBSERVABLE JUMPS
#
# Showing more entries than the detected set is useful: we want to see
# whether there is a clean scale gap between genuine change points and
# ordinary smooth trajectory curvature.
# =============================================================================

n_show = min(12, len(TIF_JUMP_ABS))

top_idx = np.argsort(
    TIF_JUMP_ABS
)[::-1][:n_show]


print("\nObservable temporal-jump audit")
print("-" * 92)

print(f"median jump norm              : {jump_median:.6e}")
print(f"MAD jump norm                 : {jump_mad:.6e}")
print(f"robust sigma                  : {jump_robust_sigma:.6e}")
print(f"detection threshold           : {TIF_JUMP_THRESHOLD:.6e}")
print(f"detected change points        : {len(TIF_CP_TIME)}")

print("\nStrongest velocity jumps:")
print(
    "rank\tjump_index\ttime\t\t"
    "absolute_jump\trelative_jump"
)

for rank, idx in enumerate(top_idx, start=1):
    print(
        f"{rank}\t"
        f"{idx}\t\t"
        f"{TIF_JUMP_TIME[idx]:.6f}\t"
        f"{TIF_JUMP_ABS[idx]:.6e}\t"
        f"{TIF_JUMP_REL[idx]:.6e}"
    )

print("\nBlindly detected change times:")
if len(TIF_CP_TIME) == 0:
    print("NONE")
else:
    for k, tc in enumerate(TIF_CP_TIME, start=1):
        print(f"change {k:2d}                    : {tc:.6f}")


# =============================================================================
# 6. CONVERT CHANGE POINTS INTO DATA-DERIVED SEGMENTS
#
# J_n compares:
#
#       interval n
#       interval n+1
#
# Therefore if J_n is a change point:
#
#       intervals 0,...,n
#
# belong to the previous segment, while
#
#       n+1,...
#
# belong to the next.
#
# No oracle stage information is used.
# =============================================================================

segment_edges = np.concatenate([
    np.array([0], dtype=int),
    TIF_CP_JUMP_INDEX + 1,
    np.array([n_intervals], dtype=int),
])

TIF_SEGMENT_SLICES = [
    slice(int(a), int(b))
    for a, b in zip(
        segment_edges[:-1],
        segment_edges[1:],
    )
]

TIF_SEGMENT_INTERVAL_COUNTS = np.array([
    sl.stop - sl.start
    for sl in TIF_SEGMENT_SLICES
])


print("\nData-derived temporal segments")
print("-" * 92)

print(
    "segment\tinterval_start\tinterval_stop\t"
    "n_intervals\tt_start\t\tt_stop"
)

for m, sl in enumerate(TIF_SEGMENT_SLICES):
    i0 = sl.start
    i1 = sl.stop

    print(
        f"{m}\t"
        f"{i0}\t\t"
        f"{i1}\t\t"
        f"{i1 - i0}\t\t"
        f"{t[i0]:.6f}\t"
        f"{t[i1]:.6f}"
    )


# =============================================================================
# 7. SAME-MIDPOINT DERIVATIVE CONSISTENCY AUDIT
#
# Fine secant centered at t_{n+1/2}:
#
#       v_dt
#       =
#       [x_{n+1} - x_n] / dt
#
# A wider 3dt secant with EXACTLY THE SAME MIDPOINT is:
#
#       v_3dt
#       =
#       [x_{n+2} - x_{n-1}] / (3 dt)
#
# for n = 1,...,N_intervals-2.
#
# For smooth dynamics both approximate F at the same midpoint.
# Their difference therefore provides an observable derivative-resolution
# audit.
#
# Any stencil touching a detected change point is excluded.
# =============================================================================

fine_interval_index = np.arange(
    1,
    n_intervals - 1,
    dtype=int,
)

V_FINE = TIF_GM_V[
    fine_interval_index
]

V_3DT = (
    X[3:] - X[:-3]
) / (3.0 * dt)

assert V_FINE.shape == V_3DT.shape


# -------------------------------------------------------------------------
# Mask all 3dt stencils that cross a detected temporal boundary.
#
# If the detected boundary is state index b, then:
#
#       interval b-1 = last interval before switch
#       interval b   = first interval after switch
#
# A 3dt stencil centered on interval n spans intervals
#
#       n-1, n, n+1.
#
# We conservatively exclude n within ±2 intervals of each boundary.
# -------------------------------------------------------------------------

TIF_DERIVATIVE_AUDIT_MASK = np.ones(
    len(fine_interval_index),
    dtype=bool,
)

detected_boundary_state_indices = (
    TIF_CP_JUMP_INDEX + 1
)

for boundary in detected_boundary_state_indices:

    TIF_DERIVATIVE_AUDIT_MASK &= (
        np.abs(
            fine_interval_index - boundary
        ) > 2
    )


derivative_difference = np.linalg.norm(
    V_FINE - V_3DT,
    axis=1,
)

derivative_relative_difference = (
    derivative_difference
    /
    np.maximum(
        np.linalg.norm(V_FINE, axis=1),
        1.0e-14,
    )
)

smooth_abs_difference = derivative_difference[
    TIF_DERIVATIVE_AUDIT_MASK
]

smooth_rel_difference = derivative_relative_difference[
    TIF_DERIVATIVE_AUDIT_MASK
]


print("\nSame-midpoint derivative-resolution audit")
print("-" * 92)

print(
    f"smooth audit points           : "
    f"{np.sum(TIF_DERIVATIVE_AUDIT_MASK)}"
)

print(
    f"median |v_dt - v_3dt|         : "
    f"{np.median(smooth_abs_difference):.6e}"
)

print(
    f"max |v_dt - v_3dt|            : "
    f"{np.max(smooth_abs_difference):.6e}"
)

print(
    f"median relative discrepancy   : "
    f"{np.median(smooth_rel_difference):.6e}"
)

print(
    f"max relative discrepancy      : "
    f"{np.max(smooth_rel_difference):.6e}"
)


# =============================================================================
# 8. BUILD STRICTLY OBSERVABLE TIF PREPROCESSING OBJECT
#
# Later inference cells should use this object rather than querying the
# synthetic construction directly.
# =============================================================================

TIF_OBS = {
    "t_state": t,
    "x_state": X,
    "dt": dt,

    "t_mid": TIF_GM_T,
    "x_mid": TIF_GM_X,
    "velocity": TIF_GM_V,

    "jump_time": TIF_JUMP_TIME,
    "jump_abs": TIF_JUMP_ABS,
    "jump_rel": TIF_JUMP_REL,
    "jump_threshold": TIF_JUMP_THRESHOLD,

    "change_times": TIF_CP_TIME.copy(),
    "change_jump_indices": TIF_CP_JUMP_INDEX.copy(),

    "segment_slices": TIF_SEGMENT_SLICES,
    "segment_interval_counts": TIF_SEGMENT_INTERVAL_COUNTS.copy(),
}


# =============================================================================
# 9. BLINDNESS CHECK
# =============================================================================

for forbidden_key in (
    "stage_edges",
    "edge_weights",
    "triads",
    "triad_strength",
    "pair_law",
    "triad_law",
    "stage_duration",
):
    assert forbidden_key not in TIF_OBS


print("\nBlind preprocessing contract")
print("-" * 92)

print("uses observable states only   : YES")
print("uses oracle topology          : NO")
print("uses oracle switch times      : NO")
print("uses number of stages         : NO")
print("uses interaction law          : NO")
print("uses repeated cycles          : NO")


print("\n" + "=" * 92)
print("CELL 1 PASSED")
print("=" * 92)

TEMPORAL INTERACTION FACTORIZATION (TIF) — CELL 1
OBSERVABLE GRADIENT MATCHING + BLIND CHANGE-POINT DETECTION

Observable input
--------------------------------------------------------------------------------------------
state samples                 : 721
transition intervals          : 720
nodes                         : 8
dt                            : 5.000000e-04

Midpoint gradient-matching observables
--------------------------------------------------------------------------------------------
midpoint states shape         : (720, 8)
velocity array shape          : (720, 8)
velocity norm range           : [3.259434e-01, 1.376344e+00]
median velocity norm          : 9.235480e-01
max |sum_i v_i|               : 1.734e-13

Observable temporal-jump audit
--------------------------------------------------------------------------------------------
median jump norm              : 1.024873e-03
MAD jump norm                 : 4.005488e-04
robust sigma                  : 5.938536e-04
detec

In [3]:
# =============================================================================
# TIF STUDY — CELL 2
# WEAK STRUCTURAL LIBRARY + ALGEBRAIC / TRAJECTORY IDENTIFIABILITY AUDIT
#
# Input
# -----
# Observable preprocessing only:
#
#     TIF_OBS
#
# Structural prior:
#
#     - all possible pair slots are admissible
#     - all possible triad slots are admissible
#     - pair law is an unknown shared conservative polynomial law
#       up to quadratic order
#     - triad law is an unknown shared conservative,
#       permutation-equivariant polynomial law up to quadratic order
#
# No true topology.
# No true coefficients.
# No true interaction law.
#
# IMPORTANT
# ---------
# This cell does NOT perform reconstruction.
#
# It first asks:
#
#   1. Is the weak structural library intrinsically identifiable?
#   2. Which directions are algebraically redundant?
#   3. How much of the identifiable structural space is excited by each
#      single-pass temporal segment?
#   4. Does the weak library reproduce the observable velocity field?
#
# =============================================================================

import numpy as np
from itertools import combinations


# =============================================================================
# 0. OBSERVABLE DATA ONLY
# =============================================================================

X_mid = np.asarray(TIF_OBS["x_mid"], dtype=float)
V_obs = np.asarray(TIF_OBS["velocity"], dtype=float)
T_mid = np.asarray(TIF_OBS["t_mid"], dtype=float)

segment_slices = TIF_OBS["segment_slices"]

assert X_mid.shape == V_obs.shape
assert X_mid.shape[1] == N


print("=" * 92)
print("TEMPORAL INTERACTION FACTORIZATION (TIF) — CELL 2")
print("WEAK STRUCTURAL LIBRARY + IDENTIFIABILITY AUDIT")
print("=" * 92)


# =============================================================================
# 1. ADMISSIBLE STRUCTURAL SLOTS
# =============================================================================

TIF_PAIR_SUPPORTS = tuple(
    combinations(range(N), 2)
)

TIF_TRIAD_SUPPORTS = tuple(
    combinations(range(N), 3)
)

N_PAIR = len(TIF_PAIR_SUPPORTS)
N_TRIAD = len(TIF_TRIAD_SUPPORTS)

assert N_PAIR == 28
assert N_TRIAD == 56


# =============================================================================
# 2. WEAK PAIRWISE FUNCTION CLASS
#
# For pair (i,j):
#
#     B1_ij:
#         flux = x_j - x_i
#
#     B2_ij:
#         flux = x_j^2 - x_i^2
#
# with conservative node action:
#
#     +flux on i
#     -flux on j
#
# The actual shared pair law is NOT specified.
#
# Any stationary conservative quadratic pair law in this basis is:
#
#     Psi_ij = c_1 B1_ij + c_2 B2_ij.
# =============================================================================

TIF_PAIR_BASIS_NAMES = (
    "pair_linear",
    "pair_quadratic",
)

L_PAIR = len(TIF_PAIR_BASIS_NAMES)


def tif_pair_basis_fields(x, i, j):
    """
    Return shape (N, 2).

    Columns:
        0 = linear conservative pair field
        1 = quadratic conservative pair field
    """
    out = np.zeros((N, L_PAIR), dtype=float)

    f1 = x[j] - x[i]
    f2 = x[j]**2 - x[i]**2

    out[i, 0] = +f1
    out[j, 0] = -f1

    out[i, 1] = +f2
    out[j, 1] = -f2

    return out


# =============================================================================
# 3. WEAK TRIAD FUNCTION CLASS
#
# For triad tau=(i,j,k), use the complete low-degree conservative,
# permutation-equivariant family generated by:
#
#     C1_i = x_i - (x_j+x_k)/2
#
#     C2_i = x_i^2 - (x_j^2+x_k^2)/2
#
#     C3_i = x_j x_k
#            - 1/2 x_i x_j
#            - 1/2 x_i x_k
#
# with cyclic permutations.
#
# We do NOT assume which combination is the true triad law.
#
# IMPORTANT:
#
# C1 and C2 may overlap algebraically with lower-order pair fields.
# We explicitly audit that below instead of pretending the 224 lifted
# coordinates are automatically independent.
# =============================================================================

TIF_TRIAD_BASIS_NAMES = (
    "triad_linear",
    "triad_quadratic_self",
    "triad_quadratic_cross",
)

L_TRIAD = len(TIF_TRIAD_BASIS_NAMES)


def tif_triad_basis_fields(x, i, j, k):
    """
    Return shape (N, 3).

    Columns form a conservative permutation-equivariant local triad class.
    """
    out = np.zeros((N, L_TRIAD), dtype=float)

    xi, xj, xk = x[i], x[j], x[k]

    # ---------------------------------------------------------------------
    # C1: linear conservative equivariant field
    # ---------------------------------------------------------------------

    out[i, 0] = xi - 0.5 * (xj + xk)
    out[j, 0] = xj - 0.5 * (xk + xi)
    out[k, 0] = xk - 0.5 * (xi + xj)

    # ---------------------------------------------------------------------
    # C2: quadratic "self-square" conservative equivariant field
    # ---------------------------------------------------------------------

    out[i, 1] = xi**2 - 0.5 * (xj**2 + xk**2)
    out[j, 1] = xj**2 - 0.5 * (xk**2 + xi**2)
    out[k, 1] = xk**2 - 0.5 * (xi**2 + xj**2)

    # ---------------------------------------------------------------------
    # C3: quadratic cross-coupled conservative equivariant field
    # ---------------------------------------------------------------------

    out[i, 2] = (
        xj * xk
        - 0.5 * xi * xj
        - 0.5 * xi * xk
    )

    out[j, 2] = (
        xk * xi
        - 0.5 * xj * xk
        - 0.5 * xj * xi
    )

    out[k, 2] = (
        xi * xj
        - 0.5 * xk * xi
        - 0.5 * xk * xj
    )

    return out


# =============================================================================
# 4. FULL LIFTED LIBRARY
#
# Ordering:
#
#     each pair support contributes 2 adjacent columns
#     each triad support contributes 3 adjacent columns
#
# Total:
#
#     28*2 + 56*3 = 224.
# =============================================================================

TIF_LIBRARY_META = []
TIF_GROUPS = []

col = 0

TIF_PAIR_COLS = {}

for support in TIF_PAIR_SUPPORTS:
    i, j = support

    group_cols = []

    for local_index, basis_name in enumerate(TIF_PAIR_BASIS_NAMES):

        TIF_LIBRARY_META.append({
            "column": col,
            "order": 2,
            "support_0b": support,
            "support_1b": (i + 1, j + 1),
            "basis_index": local_index,
            "basis_name": basis_name,
        })

        TIF_PAIR_COLS[
            (support, local_index)
        ] = col

        group_cols.append(col)
        col += 1

    TIF_GROUPS.append({
        "order": 2,
        "support_0b": support,
        "support_1b": (i + 1, j + 1),
        "columns": tuple(group_cols),
    })


TIF_TRIAD_COLS = {}

for support in TIF_TRIAD_SUPPORTS:
    i, j, k = support

    group_cols = []

    for local_index, basis_name in enumerate(TIF_TRIAD_BASIS_NAMES):

        TIF_LIBRARY_META.append({
            "column": col,
            "order": 3,
            "support_0b": support,
            "support_1b": (i + 1, j + 1, k + 1),
            "basis_index": local_index,
            "basis_name": basis_name,
        })

        TIF_TRIAD_COLS[
            (support, local_index)
        ] = col

        group_cols.append(col)
        col += 1

    TIF_GROUPS.append({
        "order": 3,
        "support_0b": support,
        "support_1b": (i + 1, j + 1, k + 1),
        "columns": tuple(group_cols),
    })


TIF_Q_LIFTED = col

assert TIF_Q_LIFTED == (
    N_PAIR * L_PAIR
    +
    N_TRIAD * L_TRIAD
)

assert TIF_Q_LIFTED == 224


def tif_library_matrix(x):
    """
    Full weak structural library at one state.

    Returns
    -------
    Phi : ndarray, shape (N, 224)
    """
    Phi = np.zeros(
        (N, TIF_Q_LIFTED),
        dtype=float,
    )

    for support in TIF_PAIR_SUPPORTS:
        i, j = support

        B = tif_pair_basis_fields(
            x,
            i,
            j,
        )

        for ell in range(L_PAIR):
            q = TIF_PAIR_COLS[
                (support, ell)
            ]
            Phi[:, q] = B[:, ell]

    for support in TIF_TRIAD_SUPPORTS:
        i, j, k = support

        B = tif_triad_basis_fields(
            x,
            i,
            j,
            k,
        )

        for ell in range(L_TRIAD):
            q = TIF_TRIAD_COLS[
                (support, ell)
            ]
            Phi[:, q] = B[:, ell]

    return Phi


print("\nWeak structural library")
print("-" * 92)
print(f"possible pair supports        : {N_PAIR}")
print(f"pair basis dimension/support  : {L_PAIR}")
print(f"possible triad supports       : {N_TRIAD}")
print(f"triad basis dimension/support : {L_TRIAD}")
print(f"total lifted coordinates      : {TIF_Q_LIFTED}")


# =============================================================================
# 5. CONSERVATION AUDIT
# =============================================================================

rng = np.random.default_rng(20260812)

max_library_conservation_error = 0.0

for _ in range(32):

    x_probe = rng.uniform(
        -0.8,
        0.8,
        size=N,
    )

    Phi = tif_library_matrix(
        x_probe
    )

    err = np.max(
        np.abs(
            np.sum(Phi, axis=0)
        )
    )

    max_library_conservation_error = max(
        max_library_conservation_error,
        err,
    )


print("\nLibrary conservation audit")
print("-" * 92)
print(
    "max columnwise |sum_i Phi_i| : "
    f"{max_library_conservation_error:.3e}"
)

assert max_library_conservation_error < 1e-12


# =============================================================================
# 6. EXACT LOWER-ORDER REDUNDANCY AUDIT
#
# For every triad (i,j,k):
#
#     C1_ijk
#       =
#       -1/2 [
#           B1_ij + B1_ik + B1_jk
#       ]
#
#     C2_ijk
#       =
#       -1/2 [
#           B2_ij + B2_ik + B2_jk
#       ]
#
# This is a structural identity, not a trajectory accident.
# =============================================================================

max_linear_reduction_error = 0.0
max_quadratic_reduction_error = 0.0

for _ in range(32):

    x_probe = rng.uniform(
        -0.8,
        0.8,
        size=N,
    )

    for i, j, k in TIF_TRIAD_SUPPORTS:

        C = tif_triad_basis_fields(
            x_probe,
            i,
            j,
            k,
        )

        pair_ij = tif_pair_basis_fields(
            x_probe,
            i,
            j,
        )

        pair_ik = tif_pair_basis_fields(
            x_probe,
            i,
            k,
        )

        pair_jk = tif_pair_basis_fields(
            x_probe,
            j,
            k,
        )

        pred_C1 = -0.5 * (
            pair_ij[:, 0]
            +
            pair_ik[:, 0]
            +
            pair_jk[:, 0]
        )

        pred_C2 = -0.5 * (
            pair_ij[:, 1]
            +
            pair_ik[:, 1]
            +
            pair_jk[:, 1]
        )

        max_linear_reduction_error = max(
            max_linear_reduction_error,
            np.linalg.norm(
                C[:, 0] - pred_C1
            ),
        )

        max_quadratic_reduction_error = max(
            max_quadratic_reduction_error,
            np.linalg.norm(
                C[:, 1] - pred_C2
            ),
        )


print("\nExact lower-order overlap audit")
print("-" * 92)

print(
    "triad-linear -> pair residual : "
    f"{max_linear_reduction_error:.3e}"
)

print(
    "triad-square -> pair residual : "
    f"{max_quadratic_reduction_error:.3e}"
)


# =============================================================================
# 7. GENERIC STRUCTURAL-RANK AUDIT
#
# This is NOT inference data.
#
# Random generic states are used only to determine whether rank loss is
# intrinsic to the weak structural library itself.
#
# This separates:
#
#     algebraic non-identifiability
#
# from:
#
#     lack of excitation along the observed trajectory.
# =============================================================================

N_GENERIC_PROBES = 64

X_generic = rng.uniform(
    -0.8,
    0.8,
    size=(N_GENERIC_PROBES, N),
)

Z_generic = np.vstack([
    tif_library_matrix(x)
    for x in X_generic
])


# -------------------------------------------------------------------------
# Column RMS scaling before singular-value diagnostics.
#
# Scaling does not alter rank, but prevents raw polynomial amplitude from
# dominating numerical conditioning.
# -------------------------------------------------------------------------

generic_col_scale = np.sqrt(
    np.mean(
        Z_generic**2,
        axis=0,
    )
)

generic_col_scale = np.maximum(
    generic_col_scale,
    1.0e-14,
)

Z_generic_scaled = (
    Z_generic
    /
    generic_col_scale[None, :]
)

s_generic = np.linalg.svd(
    Z_generic_scaled,
    compute_uv=False,
)

s_generic_rel = (
    s_generic / s_generic[0]
)

rank_generic_1e10 = int(
    np.sum(
        s_generic_rel > 1.0e-10
    )
)

rank_generic_1e12 = int(
    np.sum(
        s_generic_rel > 1.0e-12
    )
)


print("\nGeneric structural-rank audit")
print("-" * 92)

print(
    f"generic design shape          : "
    f"{Z_generic.shape}"
)

print(
    f"rank @ relative 1e-10         : "
    f"{rank_generic_1e10} / {TIF_Q_LIFTED}"
)

print(
    f"rank @ relative 1e-12         : "
    f"{rank_generic_1e12} / {TIF_Q_LIFTED}"
)

print(
    f"structural nullity @ 1e-10    : "
    f"{TIF_Q_LIFTED - rank_generic_1e10}"
)

print(
    f"largest singular value        : "
    f"{s_generic[0]:.6e}"
)

print(
    f"smallest singular value       : "
    f"{s_generic[-1]:.6e}"
)

print(
    f"smallest relative singular    : "
    f"{s_generic_rel[-1]:.6e}"
)


# =============================================================================
# 8. ORDER-BLOCK STRUCTURAL RANKS
# =============================================================================

pair_columns = np.array([
    meta["column"]
    for meta in TIF_LIBRARY_META
    if meta["order"] == 2
], dtype=int)

triad_linear_columns = np.array([
    meta["column"]
    for meta in TIF_LIBRARY_META
    if (
        meta["order"] == 3
        and meta["basis_index"] == 0
    )
], dtype=int)

triad_square_columns = np.array([
    meta["column"]
    for meta in TIF_LIBRARY_META
    if (
        meta["order"] == 3
        and meta["basis_index"] == 1
    )
], dtype=int)

triad_cross_columns = np.array([
    meta["column"]
    for meta in TIF_LIBRARY_META
    if (
        meta["order"] == 3
        and meta["basis_index"] == 2
    )
], dtype=int)


def scaled_rank(Z, rel_tol=1.0e-10):
    scale = np.sqrt(
        np.mean(Z**2, axis=0)
    )

    scale = np.maximum(
        scale,
        1.0e-14,
    )

    Zs = Z / scale[None, :]

    s = np.linalg.svd(
        Zs,
        compute_uv=False,
    )

    rank = int(
        np.sum(
            s > rel_tol * s[0]
        )
    )

    return rank


rank_pair = scaled_rank(
    Z_generic[:, pair_columns]
)

rank_pair_plus_C1 = scaled_rank(
    Z_generic[
        :,
        np.concatenate([
            pair_columns,
            triad_linear_columns,
        ])
    ]
)

rank_pair_plus_C12 = scaled_rank(
    Z_generic[
        :,
        np.concatenate([
            pair_columns,
            triad_linear_columns,
            triad_square_columns,
        ])
    ]
)

rank_pair_plus_all = scaled_rank(
    Z_generic
)


print("\nOrder-block rank decomposition")
print("-" * 92)

print(
    f"pair library                  : "
    f"{rank_pair}"
)

print(
    f"pair + triad-linear           : "
    f"{rank_pair_plus_C1}"
)

print(
    f"pair + triad-linear/square    : "
    f"{rank_pair_plus_C12}"
)

print(
    f"full library incl. cross      : "
    f"{rank_pair_plus_all}"
)


# =============================================================================
# 9. OBSERVED SINGLE-TRAJECTORY SEGMENT AUDIT
#
# For each blindly detected segment:
#
#     Z_m theta_m ~= v_m
#
# We do NOT interpret theta_m.
#
# Because the lifted library can contain an exact nullspace, the
# minimum-norm least-squares coefficient vector is not physically
# meaningful.
#
# We only inspect:
#
#     - trajectory effective rank
#     - singular spectrum
#     - model-space prediction residual
# =============================================================================

TIF_SEGMENT_DESIGNS = []
TIF_SEGMENT_TARGETS = []
TIF_SEGMENT_AUDIT = []

print("\nSingle-trajectory segment identifiability")
print("-" * 92)

print(
    "seg\trows\t"
    "rank1e-8\trank1e-10\trank1e-12\t"
    "cond(1e-10)\tLS_rel_error"
)


for m, sl in enumerate(segment_slices):

    X_m = X_mid[sl]
    V_m = V_obs[sl]

    Z_m = np.vstack([
        tif_library_matrix(x)
        for x in X_m
    ])

    y_m = V_m.reshape(-1)

    assert Z_m.shape[0] == y_m.size
    assert Z_m.shape[1] == TIF_Q_LIFTED

    # ---------------------------------------------------------------------
    # Column scaling
    # ---------------------------------------------------------------------

    scale_m = np.sqrt(
        np.mean(
            Z_m**2,
            axis=0,
        )
    )

    scale_m = np.maximum(
        scale_m,
        1.0e-14,
    )

    Zs_m = (
        Z_m
        /
        scale_m[None, :]
    )

    # ---------------------------------------------------------------------
    # Singular spectrum
    # ---------------------------------------------------------------------

    s_m = np.linalg.svd(
        Zs_m,
        compute_uv=False,
    )

    rel_m = (
        s_m / s_m[0]
    )

    rank_1e8 = int(
        np.sum(
            rel_m > 1.0e-8
        )
    )

    rank_1e10 = int(
        np.sum(
            rel_m > 1.0e-10
        )
    )

    rank_1e12 = int(
        np.sum(
            rel_m > 1.0e-12
        )
    )

    if rank_1e10 > 0:
        cond_ident = (
            s_m[0]
            /
            s_m[rank_1e10 - 1]
        )
    else:
        cond_ident = np.inf

    # ---------------------------------------------------------------------
    # Unregularized model-space consistency test
    #
    # IMPORTANT:
    # coefficients are NOT interpreted because of possible nullspaces.
    # Only the prediction residual matters here.
    # ---------------------------------------------------------------------

    coef_scaled, *_ = np.linalg.lstsq(
        Zs_m,
        y_m,
        rcond=1.0e-12,
    )

    y_hat = Zs_m @ coef_scaled

    ls_rel_error = (
        np.linalg.norm(
            y_hat - y_m
        )
        /
        np.linalg.norm(y_m)
    )

    TIF_SEGMENT_DESIGNS.append({
        "Z": Z_m,
        "Z_scaled": Zs_m,
        "column_scale": scale_m,
        "singular_values": s_m,
        "relative_singular_values": rel_m,
    })

    TIF_SEGMENT_TARGETS.append(
        y_m
    )

    TIF_SEGMENT_AUDIT.append({
        "segment": m,
        "rows": Z_m.shape[0],
        "rank_1e8": rank_1e8,
        "rank_1e10": rank_1e10,
        "rank_1e12": rank_1e12,
        "condition_1e10": cond_ident,
        "ls_relative_error": ls_rel_error,
    })

    print(
        f"{m}\t"
        f"{Z_m.shape[0]}\t"
        f"{rank_1e8}\t\t"
        f"{rank_1e10}\t\t"
        f"{rank_1e12}\t\t"
        f"{cond_ident:.3e}\t"
        f"{ls_rel_error:.3e}"
    )


# =============================================================================
# 10. SAVE LIBRARY / AUDIT OBJECTS
# =============================================================================

TIF_LIBRARY = {
    "Q_lifted": TIF_Q_LIFTED,

    "pair_supports": TIF_PAIR_SUPPORTS,
    "triad_supports": TIF_TRIAD_SUPPORTS,

    "pair_basis_names": TIF_PAIR_BASIS_NAMES,
    "triad_basis_names": TIF_TRIAD_BASIS_NAMES,

    "metadata": TIF_LIBRARY_META,
    "groups": TIF_GROUPS,

    "pair_columns": pair_columns,
    "triad_linear_columns": triad_linear_columns,
    "triad_square_columns": triad_square_columns,
    "triad_cross_columns": triad_cross_columns,

    "generic_rank_1e10": rank_generic_1e10,
    "generic_rank_1e12": rank_generic_1e12,

    "segment_audit": TIF_SEGMENT_AUDIT,
}


# =============================================================================
# 11. BLINDNESS CONTRACT
# =============================================================================

print("\nBlind structural-audit contract")
print("-" * 92)

print("uses true topology            : NO")
print("uses true coefficients        : NO")
print("uses true pair law            : NO")
print("uses true triad law           : NO")
print("uses true stage boundaries    : NO")
print("uses repeated cycles          : NO")
print("performs coefficient inference: NO")


print("\n" + "=" * 92)
print("CELL 2 PASSED")
print("=" * 92)

TEMPORAL INTERACTION FACTORIZATION (TIF) — CELL 2
WEAK STRUCTURAL LIBRARY + IDENTIFIABILITY AUDIT

Weak structural library
--------------------------------------------------------------------------------------------
possible pair supports        : 28
pair basis dimension/support  : 2
possible triad supports       : 56
triad basis dimension/support : 3
total lifted coordinates      : 224

Library conservation audit
--------------------------------------------------------------------------------------------
max columnwise |sum_i Phi_i| : 2.220e-16

Exact lower-order overlap audit
--------------------------------------------------------------------------------------------
triad-linear -> pair residual : 2.289e-16
triad-square -> pair residual : 1.272e-16

Generic structural-rank audit
--------------------------------------------------------------------------------------------
generic design shape          : (512, 224)
rank @ relative 1e-10         : 112 / 224
rank @ relative 1e-12        

In [4]:
# =============================================================================
# TIF STUDY — CELL 3
# IRREDUCIBLE QUOTIENT + BLIND STATIONARY PAIR-LAW PROFILING
#
# Purpose
# -------
# 1. Quotient out the exactly pair-reducible triad directions found in Cell 2.
#
# 2. Retain the identifiable weak structural space:
#
#       56 pair lifted directions
#       +
#       56 irreducible triad-cross directions
#
#       = 112 dimensions.
#
# 3. Exploit the stationary shared pair law:
#
#       Psi_e(theta)
#       =
#       cos(theta) B1_e
#       +
#       sin(theta) B2_e
#
#    where the overall scale is absorbed into the time-dependent edge
#    coefficient.
#
# 4. For each candidate theta:
#
#       - collapse the pair sector from 56 -> 28 columns
#       - retain 56 irreducible triad columns
#       - obtain an 84-support physical library
#       - sparsely reconstruct each temporal segment
#       - evaluate on held-out states from the SAME segment
#
# 5. Select the common theta that permits all temporal regimes to be
#    represented accurately with minimal structural complexity.
#
# NO ORACLE INFORMATION IS USED.
# =============================================================================

import numpy as np
import warnings

from sklearn.linear_model import lasso_path
from sklearn.exceptions import ConvergenceWarning


warnings.filterwarnings(
    "ignore",
    category=ConvergenceWarning,
)


# =============================================================================
# 0. OBSERVABLE DATA
# =============================================================================

X_mid = np.asarray(
    TIF_OBS["x_mid"],
    dtype=float,
)

V_obs = np.asarray(
    TIF_OBS["velocity"],
    dtype=float,
)

segment_slices = TIF_OBS["segment_slices"]


print("=" * 92)
print("TEMPORAL INTERACTION FACTORIZATION (TIF) — CELL 3")
print("IRREDUCIBLE QUOTIENT + BLIND STATIONARY-LAW PROFILING")
print("=" * 92)


# =============================================================================
# 1. IRREDUCIBLE QUOTIENT OF THE WEAK LIBRARY
#
# Cell 2 established exactly:
#
#     triad-linear  in pair span
#     triad-square  in pair span
#
# so these directions have no invariant interpretation as genuine
# three-body interactions.
#
# We therefore quotient them out.
#
# Retained lifted coordinates:
#
#     pair B1 : 28
#     pair B2 : 28
#     triad C3: 56
#
# total = 112.
# =============================================================================

TIF_Q_PAIR_LIFTED = 2 * N_PAIR
TIF_Q_TRIAD_IRR = N_TRIAD

TIF_Q_QUOTIENT = (
    TIF_Q_PAIR_LIFTED
    +
    TIF_Q_TRIAD_IRR
)

assert TIF_Q_QUOTIENT == 112


print("\nIrreducible weak structural quotient")
print("-" * 92)

print(
    f"pair lifted directions        : "
    f"{TIF_Q_PAIR_LIFTED}"
)

print(
    f"irreducible triad directions  : "
    f"{TIF_Q_TRIAD_IRR}"
)

print(
    f"quotient dimension            : "
    f"{TIF_Q_QUOTIENT}"
)


# =============================================================================
# 2. PRECOMPUTE THE THREE NECESSARY DATA-DEPENDENT BLOCKS
#
# For every observed midpoint x_n:
#
#     P1_n : N x 28
#     P2_n : N x 28
#     T3_n : N x 56
#
# This is still entirely within the weak structural class.
#
# No true interaction law is inserted.
# =============================================================================

n_mid = len(X_mid)

TIF_P1 = np.zeros(
    (n_mid, N, N_PAIR),
    dtype=float,
)

TIF_P2 = np.zeros(
    (n_mid, N, N_PAIR),
    dtype=float,
)

TIF_T3 = np.zeros(
    (n_mid, N, N_TRIAD),
    dtype=float,
)


for n, x in enumerate(X_mid):

    # ---------------------------------------------------------------------
    # Pair supports
    # ---------------------------------------------------------------------

    for q, (i, j) in enumerate(
        TIF_PAIR_SUPPORTS
    ):

        B = tif_pair_basis_fields(
            x,
            i,
            j,
        )

        TIF_P1[n, :, q] = B[:, 0]
        TIF_P2[n, :, q] = B[:, 1]

    # ---------------------------------------------------------------------
    # Irreducible triad supports
    # ---------------------------------------------------------------------

    for q, (i, j, k) in enumerate(
        TIF_TRIAD_SUPPORTS
    ):

        C = tif_triad_basis_fields(
            x,
            i,
            j,
            k,
        )

        TIF_T3[n, :, q] = C[:, 2]


# =============================================================================
# 3. GENERIC QUOTIENT-RANK AUDIT
#
# Re-use the generic states from Cell 2.
#
# The quotient should now be intrinsically full rank:
#
#       112 / 112
#
# if the structural decomposition is correct.
# =============================================================================

Zq_generic_blocks = []

for x in X_generic:

    P1 = np.column_stack([
        tif_pair_basis_fields(
            x,
            i,
            j,
        )[:, 0]
        for i, j in TIF_PAIR_SUPPORTS
    ])

    P2 = np.column_stack([
        tif_pair_basis_fields(
            x,
            i,
            j,
        )[:, 1]
        for i, j in TIF_PAIR_SUPPORTS
    ])

    T3 = np.column_stack([
        tif_triad_basis_fields(
            x,
            i,
            j,
            k,
        )[:, 2]
        for i, j, k in TIF_TRIAD_SUPPORTS
    ])

    Zq_generic_blocks.append(
        np.hstack([
            P1,
            P2,
            T3,
        ])
    )


Zq_generic = np.vstack(
    Zq_generic_blocks
)

q_scale = np.sqrt(
    np.mean(
        Zq_generic**2,
        axis=0,
    )
)

q_scale = np.maximum(
    q_scale,
    1.0e-14,
)

Zq_generic_scaled = (
    Zq_generic
    /
    q_scale[None, :]
)

sq = np.linalg.svd(
    Zq_generic_scaled,
    compute_uv=False,
)

sq_rel = sq / sq[0]

TIF_QUOTIENT_GENERIC_RANK = int(
    np.sum(
        sq_rel > 1.0e-10
    )
)


print("\nGeneric quotient-rank audit")
print("-" * 92)

print(
    f"generic quotient design       : "
    f"{Zq_generic.shape}"
)

print(
    f"rank @ relative 1e-10         : "
    f"{TIF_QUOTIENT_GENERIC_RANK} / "
    f"{TIF_Q_QUOTIENT}"
)

print(
    f"smallest relative singular    : "
    f"{sq_rel[-1]:.6e}"
)


# =============================================================================
# 4. PHYSICAL LIBRARY FOR A CANDIDATE STATIONARY PAIR LAW
#
# Scale gauge:
#
#       beta_e * c
#
# is unchanged under:
#
#       beta_e -> lambda beta_e
#       c      -> c/lambda.
#
# Therefore normalize:
#
#       ||c||_2 = 1.
#
# Parameterize the projective pair-law direction as:
#
#       c(theta) = (cos theta, sin theta)
#
# with theta in [0, pi).
#
# For fixed theta:
#
#       28 pair supports
#       +
#       56 irreducible triad supports
#
#       = 84 physical coordinates.
# =============================================================================

TIF_Q_PHYSICAL = (
    N_PAIR
    +
    N_TRIAD
)

assert TIF_Q_PHYSICAL == 84


def tif_pair_law_direction(theta_rad):

    return np.array([
        np.cos(theta_rad),
        np.sin(theta_rad),
    ])


def tif_segment_candidate_blocks(
    segment_index,
    theta_rad,
):

    sl = segment_slices[
        segment_index
    ]

    c = tif_pair_law_direction(
        theta_rad
    )

    pair_block = (
        c[0] * TIF_P1[sl]
        +
        c[1] * TIF_P2[sl]
    )

    triad_block = TIF_T3[sl]

    return (
        pair_block,
        triad_block,
        V_obs[sl],
    )


# =============================================================================
# 5. DATA-DERIVED MODEL-ACCURACY TOLERANCE
#
# Cell 1 measured the discrepancy between dt and same-midpoint 3dt
# derivative estimates.
#
# We use that purely observable resolution error to set a conservative
# threshold for saying:
#
#       "this sparse model explains the trajectory to derivative accuracy."
#
# No oracle dynamics enters.
# =============================================================================

if "smooth_rel_difference" in globals():

    derivative_resolution_floor = float(
        np.max(
            smooth_rel_difference
        )
    )

else:

    # ---------------------------------------------------------------------
    # Fallback recomputation if Cell 1 variables were cleaned manually.
    # ---------------------------------------------------------------------

    fine_index = np.arange(
        1,
        len(X_mid) - 1,
        dtype=int,
    )

    V_fine = TIF_GM_V[
        fine_index
    ]

    V_3dt = (
        TIF_DATA_X[3:]
        -
        TIF_DATA_X[:-3]
    ) / (3.0 * TIF_DT)

    mask = np.ones(
        len(fine_index),
        dtype=bool,
    )

    boundaries = (
        TIF_OBS[
            "change_jump_indices"
        ]
        + 1
    )

    for b in boundaries:

        mask &= (
            np.abs(
                fine_index - b
            ) > 2
        )

    rel = (
        np.linalg.norm(
            V_fine - V_3dt,
            axis=1,
        )
        /
        np.maximum(
            np.linalg.norm(
                V_fine,
                axis=1,
            ),
            1.0e-14,
        )
    )

    derivative_resolution_floor = float(
        np.max(
            rel[mask]
        )
    )


# Conservative allowance:
#
# derivative resolution floor ~ 1e-6 in Cell 1.
# We allow a factor 50 because held-out sparse reconstruction introduces
# conditioning and finite-sample effects beyond the raw derivative stencil.
#
TIF_PROFILE_REL_TOL = max(
    50.0 * derivative_resolution_floor,
    1.0e-5,
)


print("\nObservable model-resolution threshold")
print("-" * 92)

print(
    f"derivative resolution floor   : "
    f"{derivative_resolution_floor:.6e}"
)

print(
    f"profile validation tolerance  : "
    f"{TIF_PROFILE_REL_TOL:.6e}"
)


# =============================================================================
# 6. BLOCKED WITHIN-SEGMENT TRAIN / VALIDATION SPLIT
#
# No random splitting.
#
# For every blindly detected temporal regime:
#
#       first 75% intervals -> fit
#       final 25% intervals -> validation
#
# This deliberately tests whether a sparse model learned earlier in the
# same dynamical regime extrapolates along the trajectory.
#
# The split uses only data-derived segment boundaries.
# =============================================================================

TIF_PROFILE_SPLITS = []

for m, sl in enumerate(
    segment_slices
):

    n_int = sl.stop - sl.start

    n_fit = int(
        np.floor(
            0.75 * n_int
        )
    )

    fit_local = np.arange(
        0,
        n_fit,
        dtype=int,
    )

    val_local = np.arange(
        n_fit,
        n_int,
        dtype=int,
    )

    TIF_PROFILE_SPLITS.append({
        "fit": fit_local,
        "validation": val_local,
    })


print("\nBlocked profile split")
print("-" * 92)

print(
    "segment\tfit_intervals\tvalidation_intervals"
)

for m, split in enumerate(
    TIF_PROFILE_SPLITS
):

    print(
        f"{m}\t"
        f"{len(split['fit'])}\t\t"
        f"{len(split['validation'])}"
    )


# =============================================================================
# 7. SPARSE FIT FOR ONE SEGMENT AND ONE CANDIDATE LAW
#
# Procedure
# ---------
#
# 1. Build the 84-column candidate physical library.
# 2. RMS-standardize columns on the fit data.
# 3. Run a LASSO path.
# 4. For every selected support:
#
#       remove L1 shrinkage via post-selection OLS.
#
# 5. Evaluate the OLS-refitted model on held-out trajectory states.
#
# Selection inside one segment:
#
#     among models reaching the observable accuracy tolerance,
#     choose the one with the fewest active supports;
#
#     ties are broken by lower validation error.
#
# This is deliberately complexity-first once observational accuracy has
# been reached.
# =============================================================================

TIF_PROFILE_N_LAMBDA = 32
TIF_PROFILE_LAMBDA_RATIO_MIN = 1.0e-6


def tif_profile_one_segment(
    segment_index,
    theta_rad,
):

    pair_block, triad_block, V_m = (
        tif_segment_candidate_blocks(
            segment_index,
            theta_rad,
        )
    )

    split = TIF_PROFILE_SPLITS[
        segment_index
    ]

    fit_idx = split["fit"]
    val_idx = split["validation"]

    # ---------------------------------------------------------------------
    # interval x node x feature
    # ---------------------------------------------------------------------

    Z_full_3d = np.concatenate(
        [
            pair_block,
            triad_block,
        ],
        axis=2,
    )

    assert (
        Z_full_3d.shape[2]
        ==
        TIF_Q_PHYSICAL
    )

    # ---------------------------------------------------------------------
    # Flatten interval/node dimensions
    # ---------------------------------------------------------------------

    Z_fit = (
        Z_full_3d[fit_idx]
        .reshape(
            -1,
            TIF_Q_PHYSICAL,
        )
    )

    y_fit = (
        V_m[fit_idx]
        .reshape(-1)
    )

    Z_val = (
        Z_full_3d[val_idx]
        .reshape(
            -1,
            TIF_Q_PHYSICAL,
        )
    )

    y_val = (
        V_m[val_idx]
        .reshape(-1)
    )

    # ---------------------------------------------------------------------
    # RMS feature normalization
    # ---------------------------------------------------------------------

    scale = np.sqrt(
        np.mean(
            Z_fit**2,
            axis=0,
        )
    )

    scale = np.maximum(
        scale,
        1.0e-14,
    )

    Zs_fit = (
        Z_fit
        /
        scale[None, :]
    )

    # ---------------------------------------------------------------------
    # LASSO path
    #
    # alpha_max for sklearn objective:
    #
    #     (1 / 2n) ||y-Xw||^2
    #     +
    #     alpha ||w||_1
    # ---------------------------------------------------------------------

    alpha_max = (
        np.max(
            np.abs(
                Zs_fit.T @ y_fit
            )
        )
        /
        len(y_fit)
    )

    alphas = (
        alpha_max
        *
        np.geomspace(
            1.0,
            TIF_PROFILE_LAMBDA_RATIO_MIN,
            TIF_PROFILE_N_LAMBDA,
        )
    )

    _, coef_path_scaled, _ = (
        lasso_path(
            Zs_fit,
            y_fit,
            alphas=alphas,
            max_iter=50000,
            tol=1.0e-10,
        )
    )

    candidates = []

    for j in range(
        coef_path_scaled.shape[1]
    ):

        coef_scaled = (
            coef_path_scaled[:, j]
        )

        coef_raw = (
            coef_scaled
            /
            scale
        )

        max_coef = np.max(
            np.abs(
                coef_raw
            )
        )

        if max_coef <= 1.0e-14:
            active = np.array(
                [],
                dtype=int,
            )
        else:
            active = np.flatnonzero(
                np.abs(
                    coef_raw
                )
                >
                1.0e-7 * max_coef
            )

        # -----------------------------------------------------------------
        # Post-selection OLS removes L1 shrinkage.
        # -----------------------------------------------------------------

        if len(active) == 0:

            train_rel = 1.0
            val_rel = 1.0

            coef_refit = np.zeros(
                TIF_Q_PHYSICAL
            )

        else:

            b_active, *_ = (
                np.linalg.lstsq(
                    Z_fit[:, active],
                    y_fit,
                    rcond=1.0e-12,
                )
            )

            coef_refit = np.zeros(
                TIF_Q_PHYSICAL
            )

            coef_refit[
                active
            ] = b_active

            pred_fit = (
                Z_fit[:, active]
                @ b_active
            )

            pred_val = (
                Z_val[:, active]
                @ b_active
            )

            train_rel = (
                np.linalg.norm(
                    pred_fit - y_fit
                )
                /
                np.linalg.norm(
                    y_fit
                )
            )

            val_rel = (
                np.linalg.norm(
                    pred_val - y_val
                )
                /
                np.linalg.norm(
                    y_val
                )
            )

        candidates.append({
            "lambda_index": j,
            "alpha": alphas[j],
            "active": active,
            "n_active": len(active),
            "train_rel": train_rel,
            "val_rel": val_rel,
            "coef": coef_refit,
        })

    # ---------------------------------------------------------------------
    # Prefer the sparsest model that reaches observable accuracy.
    # ---------------------------------------------------------------------

    eligible = [
        c
        for c in candidates
        if (
            c["val_rel"]
            <= TIF_PROFILE_REL_TOL
        )
    ]

    if len(eligible) > 0:

        selected = min(
            eligible,
            key=lambda c: (
                c["n_active"],
                c["val_rel"],
            ),
        )

        reached_tolerance = True

    else:

        selected = min(
            candidates,
            key=lambda c: (
                c["val_rel"],
                c["n_active"],
            ),
        )

        reached_tolerance = False

    selected = dict(
        selected
    )

    selected[
        "reached_tolerance"
    ] = reached_tolerance

    return selected


# =============================================================================
# 8. PROFILE ONE STATIONARY PAIR-LAW ANGLE
#
# A valid global stationary law should work in ALL temporal regimes.
#
# Global criterion:
#
#     primary   : every segment reaches observable accuracy
#     secondary : minimum total active supports
#     tertiary  : minimum worst-case validation error
#     final tie : minimum mean validation error
# =============================================================================

def tif_profile_theta(
    theta_deg,
):

    theta_rad = np.deg2rad(
        theta_deg
    )

    segment_results = [
        tif_profile_one_segment(
            m,
            theta_rad,
        )
        for m in range(
            len(segment_slices)
        )
    ]

    reached_all = all(
        r["reached_tolerance"]
        for r in segment_results
    )

    total_active = int(
        np.sum([
            r["n_active"]
            for r in segment_results
        ])
    )

    val_errors = np.array([
        r["val_rel"]
        for r in segment_results
    ])

    return {
        "theta_deg": float(
            theta_deg
        ),
        "theta_rad": float(
            theta_rad
        ),
        "law_direction": (
            tif_pair_law_direction(
                theta_rad
            )
        ),
        "reached_all": reached_all,
        "total_active": total_active,
        "mean_val_error": float(
            np.mean(
                val_errors
            )
        ),
        "max_val_error": float(
            np.max(
                val_errors
            )
        ),
        "segment_results": (
            segment_results
        ),
    }


def tif_choose_profile(
    records,
):

    fully_valid = [
        r
        for r in records
        if r["reached_all"]
    ]

    if len(
        fully_valid
    ) > 0:

        best = min(
            fully_valid,
            key=lambda r: (
                r["total_active"],
                r["max_val_error"],
                r["mean_val_error"],
            ),
        )

        return best, True

    best = min(
        records,
        key=lambda r: (
            r["max_val_error"],
            r["mean_val_error"],
            r["total_active"],
        ),
    )

    return best, False


# =============================================================================
# 9. COARSE PROJECTIVE SCAN
#
# c and -c describe the same pair-law direction because the sign can be
# absorbed into beta.
#
# Therefore theta lives on RP^1 and [0,180 degrees) is sufficient.
# =============================================================================

TIF_THETA_COARSE_DEG = np.arange(
    0.0,
    180.0,
    4.0,
)

print("\nCoarse stationary-law scan")
print("-" * 92)
print(
    f"candidate directions          : "
    f"{len(TIF_THETA_COARSE_DEG)}"
)

TIF_PROFILE_COARSE = [
    tif_profile_theta(theta)
    for theta in TIF_THETA_COARSE_DEG
]

coarse_best, coarse_has_valid = (
    tif_choose_profile(
        TIF_PROFILE_COARSE
    )
)

print(
    f"coarse best theta             : "
    f"{coarse_best['theta_deg']:.3f} deg"
)

print(
    f"all segments within tolerance : "
    f"{coarse_best['reached_all']}"
)

print(
    f"total active supports         : "
    f"{coarse_best['total_active']}"
)

print(
    f"max validation error          : "
    f"{coarse_best['max_val_error']:.6e}"
)

print(
    f"mean validation error         : "
    f"{coarse_best['mean_val_error']:.6e}"
)


# =============================================================================
# 10. FINE LOCAL SCAN
#
# Refine +/- 4 degrees around the best coarse projective direction.
#
# Wrap modulo 180 degrees.
# =============================================================================

fine_raw = np.arange(
    coarse_best["theta_deg"] - 4.0,
    coarse_best["theta_deg"] + 4.0001,
    0.05,
)

TIF_THETA_FINE_DEG = np.unique(
    np.mod(
        fine_raw,
        180.0,
    )
)

print("\nFine stationary-law scan")
print("-" * 92)

print(
    f"candidate directions          : "
    f"{len(TIF_THETA_FINE_DEG)}"
)

TIF_PROFILE_FINE = [
    tif_profile_theta(theta)
    for theta in TIF_THETA_FINE_DEG
]

TIF_PROFILE_BEST, fine_has_valid = (
    tif_choose_profile(
        TIF_PROFILE_FINE
    )
)


# =============================================================================
# 11. REPORT BLINDLY LEARNED STATIONARY LAW
# =============================================================================

TIF_PAIR_LAW_DIRECTION_BLIND = (
    TIF_PROFILE_BEST[
        "law_direction"
    ]
)

TIF_PAIR_LAW_THETA_DEG_BLIND = (
    TIF_PROFILE_BEST[
        "theta_deg"
    ]
)


print("\nBlind stationary-law profile result")
print("-" * 92)

print(
    f"selected theta                : "
    f"{TIF_PAIR_LAW_THETA_DEG_BLIND:.6f} deg"
)

print(
    "normalized pair-law direction : "
    f"["
    f"{TIF_PAIR_LAW_DIRECTION_BLIND[0]: .8f}, "
    f"{TIF_PAIR_LAW_DIRECTION_BLIND[1]: .8f}"
    f"]"
)

print(
    f"all regimes reach tolerance   : "
    f"{TIF_PROFILE_BEST['reached_all']}"
)

print(
    f"total selected supports       : "
    f"{TIF_PROFILE_BEST['total_active']}"
)

print(
    f"worst validation error        : "
    f"{TIF_PROFILE_BEST['max_val_error']:.6e}"
)

print(
    f"mean validation error         : "
    f"{TIF_PROFILE_BEST['mean_val_error']:.6e}"
)


print("\nPer-segment sparse profile")
print("-" * 92)

print(
    "segment\tactive\ttrain_error\t"
    "validation_error\ttolerance"
)

for m, result in enumerate(
    TIF_PROFILE_BEST[
        "segment_results"
    ]
):

    print(
        f"{m}\t"
        f"{result['n_active']}\t"
        f"{result['train_rel']:.6e}\t"
        f"{result['val_rel']:.6e}\t"
        f"{result['reached_tolerance']}"
    )


# =============================================================================
# 12. SHOW THE BEST PROFILE DIRECTIONS
#
# No oracle comparison.
#
# We want to know whether the sparse profile has a sharply preferred
# stationary-law direction or a broad degeneracy.
# =============================================================================

all_profile_records = (
    TIF_PROFILE_COARSE
    +
    TIF_PROFILE_FINE
)

# Deduplicate approximately equal theta values.
profile_by_theta = {}

for r in all_profile_records:

    key = round(
        r["theta_deg"],
        8,
    )

    old = profile_by_theta.get(
        key
    )

    if (
        old is None
        or
        (
            r["total_active"],
            r["max_val_error"],
        )
        <
        (
            old["total_active"],
            old["max_val_error"],
        )
    ):
        profile_by_theta[
            key
        ] = r


ranked_profile = sorted(
    profile_by_theta.values(),
    key=lambda r: (
        0 if r["reached_all"] else 1,
        r["total_active"],
        r["max_val_error"],
        r["mean_val_error"],
    ),
)


print("\nTop blind stationary-law candidates")
print("-" * 92)

print(
    "rank\ttheta_deg\tall_valid\t"
    "total_active\tmax_val\t\tmean_val"
)

for rank, r in enumerate(
    ranked_profile[:12],
    start=1,
):

    print(
        f"{rank}\t"
        f"{r['theta_deg']:.4f}\t\t"
        f"{r['reached_all']}\t\t"
        f"{r['total_active']}\t\t"
        f"{r['max_val_error']:.3e}\t"
        f"{r['mean_val_error']:.3e}"
    )


# =============================================================================
# 13. SAVE TIF FACTORIZATION PROFILE
# =============================================================================

TIF_FACTOR_PROFILE = {
    "quotient_dimension": (
        TIF_Q_QUOTIENT
    ),
    "physical_dimension_given_pair_law": (
        TIF_Q_PHYSICAL
    ),
    "quotient_generic_rank": (
        TIF_QUOTIENT_GENERIC_RANK
    ),

    "validation_tolerance": (
        TIF_PROFILE_REL_TOL
    ),

    "coarse_records": (
        TIF_PROFILE_COARSE
    ),
    "fine_records": (
        TIF_PROFILE_FINE
    ),

    "best": (
        TIF_PROFILE_BEST
    ),

    "pair_law_direction": (
        TIF_PAIR_LAW_DIRECTION_BLIND.copy()
    ),

    "pair_law_theta_deg": (
        TIF_PAIR_LAW_THETA_DEG_BLIND
    ),
}


# =============================================================================
# 14. BLINDNESS CONTRACT
# =============================================================================

print("\nBlind factorization contract")
print("-" * 92)

print("uses true topology            : NO")
print("uses true coefficients        : NO")
print("uses true pair-law coefficient: NO")
print("uses true triad-law coefficient: NO")
print("uses true stage labels        : NO")
print("uses repeated cycles          : NO")
print("uses data-derived segments    : YES")
print("uses stationary-law constraint: YES")


print("\n" + "=" * 92)
print("CELL 3 PASSED")
print("=" * 92)

TEMPORAL INTERACTION FACTORIZATION (TIF) — CELL 3
IRREDUCIBLE QUOTIENT + BLIND STATIONARY-LAW PROFILING

Irreducible weak structural quotient
--------------------------------------------------------------------------------------------
pair lifted directions        : 56
irreducible triad directions  : 56
quotient dimension            : 112

Generic quotient-rank audit
--------------------------------------------------------------------------------------------
generic quotient design       : (512, 112)
rank @ relative 1e-10         : 112 / 112
smallest relative singular    : 1.767100e-01

Observable model-resolution threshold
--------------------------------------------------------------------------------------------
derivative resolution floor   : 1.324759e-06
profile validation tolerance  : 6.623793e-05

Blocked profile split
--------------------------------------------------------------------------------------------
segment	fit_intervals	validation_intervals
0	90		30
1	90		30
2	90		30

In [5]:
# =============================================================================
# TIF STUDY — CELL 4
# BLIND CROSS-WINDOW TRANSITION IDENTIFIABILITY AUDIT
#
# Purpose
# -------
# Test the transition-first inverse formulation
#
#       Y = M theta_r + S Delta_theta
#
# around every data-derived change point.
#
# We do NOT reconstruct Delta_theta yet.
# We do NOT use the synthetic oracle.
#
# Main questions:
#
#   1. Does a baseline-only model fail across the transition scan?
#
#   2. After analytically projecting out the unknown baseline theta_r,
#      how many transition directions remain observable?
#
#   3. Is the projected transition operator numerically usable?
#
# Structural representation
# -------------------------
# Use the irreducible 112-dimensional quotient established in Cells 2--3:
#
#       28 pair-linear
#       28 pair-quadratic
#       56 irreducible triad-cross
#
# No stationary interaction-law direction is assumed.
# =============================================================================

import numpy as np


# =============================================================================
# 0. OBSERVABLE DATA + WEAK STRUCTURAL QUOTIENT
# =============================================================================

t_state = np.asarray(
    TIF_OBS["t_state"],
    dtype=float,
)

X_state = np.asarray(
    TIF_OBS["x_state"],
    dtype=float,
)

dt = float(
    TIF_OBS["dt"]
)

segment_slices = TIF_OBS[
    "segment_slices"
]

boundary_state_indices = (
    np.asarray(
        TIF_OBS["change_jump_indices"],
        dtype=int,
    )
    + 1
)


# Quotient library evaluated at every midpoint:
#
#   pair B1 : 28
#   pair B2 : 28
#   triad C3: 56
#
Bq_mid = np.concatenate(
    [
        TIF_P1,
        TIF_P2,
        TIF_T3,
    ],
    axis=2,
)

n_intervals = Bq_mid.shape[0]
Q_transition = Bq_mid.shape[2]

assert Bq_mid.shape == (
    n_intervals,
    N,
    112,
)

assert Q_transition == 112


print("=" * 96)
print("TEMPORAL INTERACTION FACTORIZATION (TIF) — CELL 4")
print("BLIND CROSS-WINDOW TRANSITION IDENTIFIABILITY AUDIT")
print("=" * 96)

print("\nInput")
print("-" * 96)

print(
    f"state samples                  : "
    f"{len(X_state)}"
)

print(
    f"trajectory intervals           : "
    f"{n_intervals}"
)

print(
    f"detected boundaries            : "
    f"{len(boundary_state_indices)}"
)

print(
    f"weak quotient dimension        : "
    f"{Q_transition}"
)

print(
    f"oracle information used        : NO"
)


# =============================================================================
# 1. CHOOSE A FIXED SCAN SCALE h FROM DATA-DERIVED SEGMENTS
#
# We require every scan around every boundary to remain entirely inside
# the two adjacent stationary regimes.
#
# Choose:
#
#       h = 1/2 * shortest detected regime duration
#
# using interval counts only.
#
# For the current dataset this should give h = 0.03, but that value is
# NOT hard-coded.
# =============================================================================

segment_counts = np.array(
    [
        sl.stop - sl.start
        for sl in segment_slices
    ],
    dtype=int,
)

TIF_SCAN_H_STEPS = int(
    np.floor(
        0.5 * np.min(segment_counts)
    )
)

TIF_SCAN_H = (
    TIF_SCAN_H_STEPS * dt
)

assert TIF_SCAN_H_STEPS >= 2


print("\nFixed-h scan geometry")
print("-" * 96)

print(
    f"shortest detected regime       : "
    f"{np.min(segment_counts)} intervals"
)

print(
    f"scan length                    : "
    f"{TIF_SCAN_H_STEPS} intervals"
)

print(
    f"h                              : "
    f"{TIF_SCAN_H:.6f}"
)

print(
    f"scan positions / boundary      : "
    f"{TIF_SCAN_H_STEPS + 1}"
)


# =============================================================================
# 2. CUMULATIVE INTEGRAL OF THE WEAK LIBRARY ALONG THE OBSERVED TRAJECTORY
#
# For interval n:
#
#       int_{t_n}^{t_{n+1}} B(x(t)) dt
#
# is approximated with midpoint quadrature:
#
#       dt * B(x_mid,n).
#
# Cell 1 established that dt is already deep in the local derivative
# regime, so midpoint quadrature should be extremely accurate here.
#
# Prefix array:
#
#       LIB_INT_PREFIX[k]
#
# represents the integral from state time 0 up to state index k.
#
# Therefore:
#
#       integral[a:b]
#       =
#       LIB_INT_PREFIX[b] - LIB_INT_PREFIX[a].
# =============================================================================

TIF_LIB_INT_PREFIX = np.zeros(
    (
        n_intervals + 1,
        N,
        Q_transition,
    ),
    dtype=float,
)

TIF_LIB_INT_PREFIX[1:] = np.cumsum(
    dt * Bq_mid,
    axis=0,
)


def tif_integrated_library(a, b):
    """
    Approximate integral of B(x(t)) dt from state index a to b.

    Covers trajectory intervals:
        a, a+1, ..., b-1
    """
    return (
        TIF_LIB_INT_PREFIX[b]
        -
        TIF_LIB_INT_PREFIX[a]
    )


# =============================================================================
# 3. NUMERICAL-RANK HELPERS
# =============================================================================

def tif_column_scale(A):
    """
    Euclidean column scale.
    """
    scale = np.linalg.norm(
        A,
        axis=0,
    )

    return np.maximum(
        scale,
        1.0e-14,
    )


def tif_singular_audit(
    A,
    external_scale=None,
):
    """
    Singular-spectrum audit after column scaling.

    If external_scale is given, it is used instead of renormalizing the
    matrix itself. This is important for projected designs: we want to
    preserve how much of each original transition direction survives
    nuisance projection.
    """

    if external_scale is None:
        scale = tif_column_scale(A)
    else:
        scale = np.asarray(
            external_scale,
            dtype=float,
        )

    As = (
        A
        /
        scale[None, :]
    )

    s = np.linalg.svd(
        As,
        compute_uv=False,
    )

    if len(s) == 0 or s[0] == 0.0:

        rel = np.zeros_like(s)

    else:

        rel = s / s[0]

    ranks = {
        tol: int(
            np.sum(
                rel > tol
            )
        )
        for tol in (
            1.0e-8,
            1.0e-10,
            1.0e-12,
        )
    }

    return {
        "scale": scale,
        "singular_values": s,
        "relative_singular_values": rel,
        "ranks": ranks,
    }


# =============================================================================
# 4. BUILD ONE FIXED-h SCAN AROUND ONE DETECTED BOUNDARY
#
# Let b be the boundary state index.
#
# A window has fixed length h_steps and starting state index a.
#
# Scan:
#
#       a = b-h_steps, ..., b
#
# so the first window is purely pre-transition and the last is purely
# post-transition.
#
# For window j:
#
#       Q^-_j = integral[a:b]
#       Q^+_j = integral[b:e]
#
#       e = a + h_steps.
#
# Exact structural model:
#
#       Delta x_j
#       =
#       Q^-_j theta_r
#       +
#       Q^+_j theta_s
#
# Define:
#
#       Delta_theta = theta_s - theta_r
#
# giving:
#
#       Delta x_j
#       =
#       (Q^-_j + Q^+_j) theta_r
#       +
#       Q^+_j Delta_theta.
#
# Hence:
#
#       M_j = Q^-_j + Q^+_j
#       S_j = Q^+_j.
# =============================================================================

def tif_build_transition_scan(
    boundary_index,
    h_steps,
):

    b = int(
        boundary_index
    )

    start_first = (
        b - h_steps
    )

    start_last = b

    assert start_first >= 0
    assert (
        start_last + h_steps
        <= n_intervals
    )

    starts = np.arange(
        start_first,
        start_last + 1,
        dtype=int,
    )

    M_blocks = []
    S_blocks = []
    y_blocks = []

    lambdas = []
    ends = []

    for a in starts:

        e = int(
            a + h_steps
        )

        Q_minus = tif_integrated_library(
            a,
            b,
        )

        Q_plus = tif_integrated_library(
            b,
            e,
        )

        M_j = (
            Q_minus
            +
            Q_plus
        )

        S_j = Q_plus

        y_j = (
            X_state[e]
            -
            X_state[a]
        )

        # Fraction of the window lying after the boundary.
        lambda_j = (
            (e - b)
            /
            h_steps
        )

        M_blocks.append(
            M_j
        )

        S_blocks.append(
            S_j
        )

        y_blocks.append(
            y_j
        )

        lambdas.append(
            lambda_j
        )

        ends.append(
            e
        )

    M = np.vstack(
        M_blocks
    )

    S = np.vstack(
        S_blocks
    )

    y = np.concatenate(
        y_blocks
    )

    return {
        "boundary_state_index": b,
        "boundary_time": t_state[b],

        "starts": starts,
        "ends": np.asarray(
            ends,
            dtype=int,
        ),

        "lambda": np.asarray(
            lambdas,
            dtype=float,
        ),

        "M": M,
        "S": S,
        "y": y,
    }


# =============================================================================
# 5. BASELINE-NUISANCE PROJECTION
#
# Model:
#
#       y = M theta_r + S Delta_theta.
#
# We remove theta_r analytically.
#
# Let U_M span col(M). Then:
#
#       P_M_perp y
#       =
#       y - U_M U_M^T y
#
# and likewise
#
#       P_M_perp S
#       =
#       S - U_M U_M^T S.
#
# No explicit (large) projector matrix is constructed.
# =============================================================================

def tif_project_out_baseline(
    M,
    S,
    y,
    rank_tol=1.0e-12,
):

    # ---------------------------------------------------------------------
    # Scale M only for numerical rank determination.
    # Scaling does not alter its column space.
    # ---------------------------------------------------------------------

    M_scale = tif_column_scale(
        M
    )

    Ms = (
        M
        /
        M_scale[None, :]
    )

    U, s, _ = np.linalg.svd(
        Ms,
        full_matrices=False,
    )

    if s[0] == 0.0:
        rank_M = 0
    else:
        rank_M = int(
            np.sum(
                s / s[0]
                > rank_tol
            )
        )

    U_r = U[
        :,
        :rank_M
    ]

    if rank_M == 0:

        y_res = y.copy()
        S_res = S.copy()

    else:

        y_res = (
            y
            -
            U_r @ (
                U_r.T @ y
            )
        )

        S_res = (
            S
            -
            U_r @ (
                U_r.T @ S
            )
        )

    return {
        "rank_M_projection": rank_M,
        "U_M": U_r,
        "y_res": y_res,
        "S_res": S_res,
    }


# =============================================================================
# 6. AUDIT EACH DETECTED TRANSITION
# =============================================================================

TIF_TRANSITION_SCAN_AUDIT = []


print("\nCross-window transition audit")
print("-" * 96)

print(
    "bdry  time      rows  "
    "rank(M)  rank([M,S])  "
    "rank(PMperp S)  "
    "base_err    joint_err"
)


for boundary_number, b in enumerate(
    boundary_state_indices,
    start=1,
):

    scan = tif_build_transition_scan(
        b,
        TIF_SCAN_H_STEPS,
    )

    M = scan["M"]
    S = scan["S"]
    y = scan["y"]

    # ---------------------------------------------------------------------
    # Basic ranks
    # ---------------------------------------------------------------------

    audit_M = tif_singular_audit(
        M
    )

    audit_joint = tif_singular_audit(
        np.hstack(
            [M, S]
        )
    )

    # ---------------------------------------------------------------------
    # Project baseline nuisance directions away
    # ---------------------------------------------------------------------

    projected = tif_project_out_baseline(
        M,
        S,
        y,
        rank_tol=1.0e-12,
    )

    y_res = projected[
        "y_res"
    ]

    S_res = projected[
        "S_res"
    ]

    # ---------------------------------------------------------------------
    # IMPORTANT:
    #
    # Scale projected S by the ORIGINAL S column norms.
    #
    # If a transition direction is almost entirely absorbed by M, we do
    # not want to hide that fact by renormalizing the tiny residual back
    # to unit size.
    # ---------------------------------------------------------------------

    S_original_scale = tif_column_scale(
        S
    )

    audit_S_res = tif_singular_audit(
        S_res,
        external_scale=S_original_scale,
    )

    # ---------------------------------------------------------------------
    # Per-column transition visibility
    #
    #       ||P_M_perp S_q|| / ||S_q||
    #
    # 0 -> transition coordinate is completely confounded with baseline.
    # 1 -> transition coordinate is orthogonal to baseline space.
    # ---------------------------------------------------------------------

    S_norm = np.linalg.norm(
        S,
        axis=0,
    )

    S_res_norm = np.linalg.norm(
        S_res,
        axis=0,
    )

    visibility = (
        S_res_norm
        /
        np.maximum(
            S_norm,
            1.0e-14,
        )
    )

    # ---------------------------------------------------------------------
    # Baseline-only model consistency
    # ---------------------------------------------------------------------

    M_scale = audit_M[
        "scale"
    ]

    Ms = (
        M
        /
        M_scale[None, :]
    )

    coef_M, *_ = np.linalg.lstsq(
        Ms,
        y,
        rcond=1.0e-12,
    )

    pred_M = (
        Ms @ coef_M
    )

    baseline_rel_error = (
        np.linalg.norm(
            pred_M - y
        )
        /
        np.linalg.norm(y)
    )

    # ---------------------------------------------------------------------
    # Joint unrestricted consistency
    #
    # This is NOT structural inference.
    # It only checks whether the weak quotient library can represent the
    # observed cross-window increments.
    # ---------------------------------------------------------------------

    J = np.hstack(
        [M, S]
    )

    J_scale = tif_column_scale(
        J
    )

    Js = (
        J
        /
        J_scale[None, :]
    )

    coef_joint, *_ = np.linalg.lstsq(
        Js,
        y,
        rcond=1.0e-12,
    )

    pred_joint = (
        Js @ coef_joint
    )

    joint_rel_error = (
        np.linalg.norm(
            pred_joint - y
        )
        /
        np.linalg.norm(y)
    )

    # ---------------------------------------------------------------------
    # How much of the baseline-unexplained signal can the projected
    # transition space explain?
    # ---------------------------------------------------------------------

    if np.linalg.norm(
        y_res
    ) > 1.0e-14:

        Sres_scale = np.maximum(
            S_original_scale,
            1.0e-14,
        )

        Sres_s = (
            S_res
            /
            Sres_scale[None, :]
        )

        coef_delta, *_ = np.linalg.lstsq(
            Sres_s,
            y_res,
            rcond=1.0e-12,
        )

        pred_res = (
            Sres_s
            @ coef_delta
        )

        projected_fit_rel_error = (
            np.linalg.norm(
                pred_res - y_res
            )
            /
            np.linalg.norm(
                y_res
            )
        )

    else:

        projected_fit_rel_error = 0.0

    result = {
        **scan,

        "rank_M_1e8":
            audit_M["ranks"][1.0e-8],

        "rank_M_1e10":
            audit_M["ranks"][1.0e-10],

        "rank_M_1e12":
            audit_M["ranks"][1.0e-12],

        "rank_joint_1e8":
            audit_joint["ranks"][1.0e-8],

        "rank_joint_1e10":
            audit_joint["ranks"][1.0e-10],

        "rank_joint_1e12":
            audit_joint["ranks"][1.0e-12],

        "rank_transition_1e8":
            audit_S_res["ranks"][1.0e-8],

        "rank_transition_1e10":
            audit_S_res["ranks"][1.0e-10],

        "rank_transition_1e12":
            audit_S_res["ranks"][1.0e-12],

        "transition_singular_values":
            audit_S_res["singular_values"],

        "transition_relative_singular_values":
            audit_S_res["relative_singular_values"],

        "transition_visibility":
            visibility,

        "baseline_relative_error":
            baseline_rel_error,

        "joint_relative_error":
            joint_rel_error,

        "projected_fit_relative_error":
            projected_fit_rel_error,

        "y_projected":
            y_res,

        "S_projected":
            S_res,
    }

    TIF_TRANSITION_SCAN_AUDIT.append(
        result
    )

    print(
        f"{boundary_number:>4d}  "
        f"{scan['boundary_time']:.6f}  "
        f"{len(y):>4d}  "
        f"{audit_M['ranks'][1.0e-10]:>7d}  "
        f"{audit_joint['ranks'][1.0e-10]:>11d}  "
        f"{audit_S_res['ranks'][1.0e-10]:>14d}  "
        f"{baseline_rel_error:.3e}  "
        f"{joint_rel_error:.3e}"
    )


# =============================================================================
# 7. TRANSITION-VISIBILITY REPORT
# =============================================================================

print("\nProjected transition visibility")
print("-" * 96)

print(
    "bdry  time      "
    "rank1e-8  rank1e-10  rank1e-12  "
    "vis_min    vis_med    vis_max    proj_fit_err"
)


for boundary_number, result in enumerate(
    TIF_TRANSITION_SCAN_AUDIT,
    start=1,
):

    vis = result[
        "transition_visibility"
    ]

    print(
        f"{boundary_number:>4d}  "
        f"{result['boundary_time']:.6f}  "
        f"{result['rank_transition_1e8']:>8d}  "
        f"{result['rank_transition_1e10']:>10d}  "
        f"{result['rank_transition_1e12']:>10d}  "
        f"{np.min(vis):.3e}  "
        f"{np.median(vis):.3e}  "
        f"{np.max(vis):.3e}  "
        f"{result['projected_fit_relative_error']:.3e}"
    )


# =============================================================================
# 8. VISIBILITY BY STRUCTURAL SECTOR
#
# Still completely blind:
#
#       columns 0:28     pair-linear
#       columns 28:56    pair-quadratic
#       columns 56:112   irreducible triad
#
# This tells us whether one structural sector is systematically swallowed
# by the unknown baseline.
# =============================================================================

sector_slices = {
    "pair_linear": slice(
        0,
        28,
    ),
    "pair_quadratic": slice(
        28,
        56,
    ),
    "triad_irreducible": slice(
        56,
        112,
    ),
}


print("\nMedian transition visibility by structural sector")
print("-" * 96)

print(
    "bdry  pair_linear  pair_quadratic  triad_irreducible"
)


for boundary_number, result in enumerate(
    TIF_TRANSITION_SCAN_AUDIT,
    start=1,
):

    vis = result[
        "transition_visibility"
    ]

    vals = {
        name: np.median(
            vis[sl]
        )
        for name, sl in sector_slices.items()
    }

    print(
        f"{boundary_number:>4d}  "
        f"{vals['pair_linear']:.3e}     "
        f"{vals['pair_quadratic']:.3e}        "
        f"{vals['triad_irreducible']:.3e}"
    )


# =============================================================================
# 9. SAVE BLIND CROSS-WINDOW OBJECT
# =============================================================================

TIF_TRANSITION_SCAN = {
    "h_steps":
        TIF_SCAN_H_STEPS,

    "h":
        TIF_SCAN_H,

    "quotient_dimension":
        Q_transition,

    "boundary_state_indices":
        boundary_state_indices.copy(),

    "boundary_times":
        t_state[
            boundary_state_indices
        ].copy(),

    "audits":
        TIF_TRANSITION_SCAN_AUDIT,

    "oracle_used":
        False,
}


# =============================================================================
# 10. BLINDNESS CONTRACT
# =============================================================================

print("\nBlind transition-audit contract")
print("-" * 96)

print("uses detected change points   : YES")
print("uses weak structural class    : YES")
print("uses true topology            : NO")
print("uses true changed supports    : NO")
print("uses true edge weights        : NO")
print("uses true interaction law     : NO")
print("performs sparse inference     : NO")
print("performs oracle comparison    : NO")


print("\n" + "=" * 96)
print("CELL 4 PASSED")
print("=" * 96)

TEMPORAL INTERACTION FACTORIZATION (TIF) — CELL 4
BLIND CROSS-WINDOW TRANSITION IDENTIFIABILITY AUDIT

Input
------------------------------------------------------------------------------------------------
state samples                  : 721
trajectory intervals           : 720
detected boundaries            : 5
weak quotient dimension        : 112
oracle information used        : NO

Fixed-h scan geometry
------------------------------------------------------------------------------------------------
shortest detected regime       : 120 intervals
scan length                    : 60 intervals
h                              : 0.030000
scan positions / boundary      : 61

Cross-window transition audit
------------------------------------------------------------------------------------------------
bdry  time      rows  rank(M)  rank([M,S])  rank(PMperp S)  base_err    joint_err
   1  0.060000   488       35           35             112  7.251e-13  2.040e-14
   2  0.120000   488       41 

In [6]:
# =============================================================================
# TIF STUDY — CELL 5
# TWO-SIDED TRANSITION-COORDINATE IDENTIFIABILITY AUDIT
#
# Cell 4 result:
# ----------------
# Fixed-h integrated crossing windows almost completely confounded
#
#       baseline theta_r
#
# and
#
#       transition Delta_theta.
#
# The endpoint-window compression therefore discarded useful regime-local
# information.
#
# Here we return to the dense gradient-matching observations themselves.
#
# For one detected transition:
#
#       left : v = B(x) theta_r
#       right: v = B(x) (theta_r + Delta_theta)
#
# Stack:
#
#           [B_left ]             [0      ]
#       y = [        ] theta_r +  [       ] Delta_theta
#           [B_right]             [B_right]
#
#       y = M theta_r + S Delta_theta.
#
# We ask ONLY whether Delta_theta contributes directions independent of
# the unknown baseline theta_r.
#
# No oracle.
# No sparse solver.
# =============================================================================

import numpy as np


# =============================================================================
# 0. OBSERVABLE INPUT
# =============================================================================

V = np.asarray(
    TIF_OBS["velocity"],
    dtype=float,
)

boundaries = (
    np.asarray(
        TIF_OBS["change_jump_indices"],
        dtype=int,
    )
    + 1
)

Q = Bq_mid.shape[2]

assert Q == 112
assert V.shape[0] == Bq_mid.shape[0]


print("=" * 100)
print("TIF — CELL 5")
print("TWO-SIDED TRANSITION-COORDINATE IDENTIFIABILITY AUDIT")
print("=" * 100)


# =============================================================================
# 1. LOCAL RADIUS
#
# Use the same data-derived scale as Cell 4:
#
#       half of the shortest detected regime.
#
# Thus each boundary uses:
#
#       h samples immediately before the transition
#       h samples immediately after the transition.
#
# No adjacent change point is crossed.
# =============================================================================

H = TIF_SCAN_H_STEPS

print("\nTwo-sided geometry")
print("-" * 100)
print(f"intervals on each side         : {H}")
print(f"time span on each side         : {H * dt:.6f}")
print(f"total intervals / transition   : {2 * H}")
print(f"quotient coordinates           : {Q}")


# =============================================================================
# 2. CONSISTENT NUMERICAL-RANK FUNCTION
#
# IMPORTANT:
#
# All matrices are column-scaled BEFORE comparing ranks.
#
# For [M,S], M and S are independently scaled by their original column
# norms. We never renormalize a tiny projected residual by its own size.
# =============================================================================

def tif_rank_from_scaled_matrix(A, rel_tol=1.0e-10):
    s = np.linalg.svd(
        A,
        compute_uv=False,
    )

    if len(s) == 0 or s[0] == 0.0:
        return 0, s

    rank = int(
        np.sum(
            s > rel_tol * s[0]
        )
    )

    return rank, s


def tif_safe_colnorm(A):
    return np.maximum(
        np.linalg.norm(A, axis=0),
        1.0e-14,
    )


# =============================================================================
# 3. BUILD TWO-SIDED DESIGN
# =============================================================================

def tif_build_two_sided_transition(boundary, H):

    b = int(boundary)

    left = np.arange(
        b - H,
        b,
        dtype=int,
    )

    right = np.arange(
        b,
        b + H,
        dtype=int,
    )

    assert left[0] >= 0
    assert right[-1] < len(V)

    B_left = (
        Bq_mid[left]
        .reshape(-1, Q)
    )

    B_right = (
        Bq_mid[right]
        .reshape(-1, Q)
    )

    y_left = (
        V[left]
        .reshape(-1)
    )

    y_right = (
        V[right]
        .reshape(-1)
    )

    # Baseline acts on both sides.
    M = np.vstack([
        B_left,
        B_right,
    ])

    # Transition acts only after the boundary.
    S = np.vstack([
        np.zeros_like(B_left),
        B_right,
    ])

    y = np.concatenate([
        y_left,
        y_right,
    ])

    return {
        "boundary": b,
        "time": t_state[b],
        "left_indices": left,
        "right_indices": right,
        "M": M,
        "S": S,
        "y": y,
    }


# =============================================================================
# 4. PROJECT OUT BASELINE
# =============================================================================

def tif_two_sided_audit(scan):

    M = scan["M"]
    S = scan["S"]
    y = scan["y"]

    # ---------------------------------------------------------------------
    # Independent column scalings.
    # ---------------------------------------------------------------------

    scale_M = tif_safe_colnorm(M)
    scale_S = tif_safe_colnorm(S)

    Ms = M / scale_M[None, :]
    Ss = S / scale_S[None, :]

    # ---------------------------------------------------------------------
    # Baseline rank and basis.
    # ---------------------------------------------------------------------

    U, sM, _ = np.linalg.svd(
        Ms,
        full_matrices=False,
    )

    if sM[0] == 0.0:
        rank_M = 0
    else:
        rank_M = int(
            np.sum(
                sM > 1.0e-10 * sM[0]
            )
        )

    U_M = U[:, :rank_M]

    # ---------------------------------------------------------------------
    # Orthogonal nuisance elimination.
    # ---------------------------------------------------------------------

    if rank_M == 0:

        S_perp = Ss.copy()
        y_perp = y.copy()

    else:

        S_perp = (
            Ss
            -
            U_M @ (
                U_M.T @ Ss
            )
        )

        y_perp = (
            y
            -
            U_M @ (
                U_M.T @ y
            )
        )

    # ---------------------------------------------------------------------
    # Joint rank.
    #
    # Since Ms and Ss are already independently normalized, this gives
    # the meaningful rank increase contributed by the transition block.
    # ---------------------------------------------------------------------

    Joint = np.hstack([
        Ms,
        Ss,
    ])

    rank_joint, sJoint = (
        tif_rank_from_scaled_matrix(
            Joint,
            rel_tol=1.0e-10,
        )
    )

    rank_gain = (
        rank_joint - rank_M
    )

    # ---------------------------------------------------------------------
    # Absolute projected singular values.
    #
    # Compare to scale 1 because Ss columns had unit norm before
    # projection.
    #
    # Therefore a singular value ~1e-12 genuinely means numerical noise;
    # we do NOT normalize S_perp by its own largest singular value.
    # ---------------------------------------------------------------------

    s_perp = np.linalg.svd(
        S_perp,
        compute_uv=False,
    )

    ranks_perp = {
        tol: int(
            np.sum(
                s_perp > tol
            )
        )
        for tol in (
            1.0e-6,
            1.0e-8,
            1.0e-10,
            1.0e-12,
        )
    }

    # ---------------------------------------------------------------------
    # Columnwise visibility:
    #
    # Ss columns have unit norm, so ||S_perp[:,q]|| directly measures
    # surviving transition visibility.
    # ---------------------------------------------------------------------

    visibility = np.linalg.norm(
        S_perp,
        axis=0,
    )

    # ---------------------------------------------------------------------
    # Baseline-only prediction.
    # ---------------------------------------------------------------------

    coef_base, *_ = np.linalg.lstsq(
        Ms,
        y,
        rcond=1.0e-12,
    )

    pred_base = Ms @ coef_base

    base_error = (
        np.linalg.norm(
            y - pred_base
        )
        /
        np.linalg.norm(y)
    )

    # ---------------------------------------------------------------------
    # Joint unrestricted prediction.
    # ---------------------------------------------------------------------

    coef_joint, *_ = np.linalg.lstsq(
        Joint,
        y,
        rcond=1.0e-12,
    )

    pred_joint = (
        Joint @ coef_joint
    )

    joint_error = (
        np.linalg.norm(
            y - pred_joint
        )
        /
        np.linalg.norm(y)
    )

    # ---------------------------------------------------------------------
    # How much of the baseline-unexplained signal lies in transition
    # space?
    # ---------------------------------------------------------------------

    if np.linalg.norm(y_perp) > 1.0e-14:

        coef_transition, *_ = np.linalg.lstsq(
            S_perp,
            y_perp,
            rcond=1.0e-12,
        )

        residual_transition = (
            y_perp
            -
            S_perp @ coef_transition
        )

        projected_error = (
            np.linalg.norm(
                residual_transition
            )
            /
            np.linalg.norm(
                y_perp
            )
        )

    else:

        projected_error = 0.0

    return {
        **scan,

        "rank_M": rank_M,
        "rank_joint": rank_joint,
        "rank_gain": rank_gain,

        "s_perp": s_perp,
        "ranks_perp": ranks_perp,

        "visibility": visibility,

        "base_error": base_error,
        "joint_error": joint_error,
        "projected_error": projected_error,

        "S_perp": S_perp,
        "y_perp": y_perp,
    }


# =============================================================================
# 5. RUN ALL FIVE BLIND TRANSITIONS
# =============================================================================

TIF_TWO_SIDED_AUDIT = []

print("\nTwo-sided transition audit")
print("-" * 100)

print(
    "bdry  time      rows  "
    "rank(M)  rank([M,S])  gain   "
    "rank_perp@1e-8  "
    "base_err    joint_err"
)


for k, b in enumerate(
    boundaries,
    start=1,
):

    scan = tif_build_two_sided_transition(
        b,
        H,
    )

    result = tif_two_sided_audit(
        scan
    )

    TIF_TWO_SIDED_AUDIT.append(
        result
    )

    print(
        f"{k:>4d}  "
        f"{result['time']:.6f}  "
        f"{len(result['y']):>4d}  "
        f"{result['rank_M']:>7d}  "
        f"{result['rank_joint']:>11d}  "
        f"{result['rank_gain']:>4d}   "
        f"{result['ranks_perp'][1.0e-8]:>14d}  "
        f"{result['base_error']:.3e}  "
        f"{result['joint_error']:.3e}"
    )


# =============================================================================
# 6. VISIBILITY REPORT
# =============================================================================

print("\nProjected transition visibility")
print("-" * 100)

print(
    "bdry  "
    "vis_min    vis_med    vis_max    "
    "rank@1e-6  rank@1e-8  rank@1e-10  "
    "proj_fit_err"
)


for k, result in enumerate(
    TIF_TWO_SIDED_AUDIT,
    start=1,
):

    vis = result[
        "visibility"
    ]

    rp = result[
        "ranks_perp"
    ]

    print(
        f"{k:>4d}  "
        f"{np.min(vis):.3e}  "
        f"{np.median(vis):.3e}  "
        f"{np.max(vis):.3e}  "
        f"{rp[1.0e-6]:>9d}  "
        f"{rp[1.0e-8]:>9d}  "
        f"{rp[1.0e-10]:>10d}  "
        f"{result['projected_error']:.3e}"
    )


# =============================================================================
# 7. STRUCTURAL-SECTOR VISIBILITY
# =============================================================================

sector_slices = {
    "pair_linear": slice(0, 28),
    "pair_quadratic": slice(28, 56),
    "triad_irreducible": slice(56, 112),
}


print("\nMedian visibility by structural sector")
print("-" * 100)

print(
    "bdry  pair_linear  pair_quadratic  triad_irreducible"
)


for k, result in enumerate(
    TIF_TWO_SIDED_AUDIT,
    start=1,
):

    vis = result[
        "visibility"
    ]

    vals = {
        name: np.median(
            vis[sl]
        )
        for name, sl in sector_slices.items()
    }

    print(
        f"{k:>4d}  "
        f"{vals['pair_linear']:.3e}     "
        f"{vals['pair_quadratic']:.3e}        "
        f"{vals['triad_irreducible']:.3e}"
    )


# =============================================================================
# 8. SAVE
# =============================================================================

TIF_TWO_SIDED_TRANSITION = {
    "H_steps": H,
    "H_time": H * dt,
    "audits": TIF_TWO_SIDED_AUDIT,
    "oracle_used": False,
}


# =============================================================================
# 9. CONTRACT
# =============================================================================

print("\nBlindness contract")
print("-" * 100)

print("uses detected boundaries      : YES")
print("uses full dense trajectory    : YES")
print("uses weak quotient library    : YES")
print("uses true changed supports    : NO")
print("uses true topology            : NO")
print("uses true interaction law     : NO")
print("uses sparse inference         : NO")
print("uses oracle                   : NO")


print("\n" + "=" * 100)
print("CELL 5 PASSED")
print("=" * 100)

TIF — CELL 5
TWO-SIDED TRANSITION-COORDINATE IDENTIFIABILITY AUDIT

Two-sided geometry
----------------------------------------------------------------------------------------------------
intervals on each side         : 60
time span on each side         : 0.030000
total intervals / transition   : 120
quotient coordinates           : 112

Two-sided transition audit
----------------------------------------------------------------------------------------------------
bdry  time      rows  rank(M)  rank([M,S])  gain   rank_perp@1e-8  base_err    joint_err
   1  0.060000   960       59           63     4                7  1.706e-05  2.252e-13
   2  0.120000   960       67           69     2                7  1.358e-05  1.859e-13
   3  0.180000   960       66           70     4                7  2.413e-05  3.804e-13
   4  0.240000   960       69           70     1                7  4.873e-05  5.553e-13
   5  0.300000   960       70           70     0                7  2.284e-05  2.584e-13

P

In [7]:
# =============================================================================
# TIF STUDY — CELL 6
# ORACLE-RESTRICTED TRANSITION IDENTIFIABILITY DIAGNOSTIC
#
# IMPORTANT
# ---------
# This is the FIRST cell that intentionally opens the synthetic oracle
# after the blind transition operator has been constructed.
#
# The oracle is used ONLY for diagnosis:
#
#     - What is the true changed support?
#     - Does that true sparse transition survive baseline projection?
#     - Is the true changed support identifiable if the shared law is known?
#     - Is it locally identifiable when the shared law is unknown but shared?
#
# NO support selection.
# NO LASSO.
# NO oracle information is fed into an inference algorithm.
# =============================================================================

import numpy as np


# =============================================================================
# 0. OPEN SYNTHETIC ORACLE FOR DIAGNOSTICS ONLY
# =============================================================================

oracle_stage_edges_1b = TIF_ORACLE[
    "stage_edges_1b"
]

oracle_weights_1b = TIF_ORACLE[
    "edge_weights_1b"
]

oracle_pair_law = np.asarray(
    TIF_ORACLE[
        "pair_law_coefficients"
    ],
    dtype=float,
)

oracle_triads_1b = TIF_ORACLE[
    "triads_1b"
]

oracle_triad_strength = float(
    TIF_ORACLE[
        "triad_strength"
    ]
)


assert len(oracle_pair_law) == 2


# Canonical zero-based weight dictionary
oracle_weight_map = {
    tuple(
        sorted(
            (i - 1, j - 1)
        )
    ): float(w)
    for (i, j), w
    in oracle_weights_1b.items()
}

oracle_stage_supports = [
    {
        tuple(
            sorted(
                (i - 1, j - 1)
            )
        )
        for i, j in stage
    }
    for stage in oracle_stage_edges_1b
]

oracle_triads = {
    tuple(
        sorted(
            (i - 1, j - 1, k - 1)
        )
    )
    for i, j, k in oracle_triads_1b
}


print("=" * 104)
print("TIF — CELL 6")
print("ORACLE-RESTRICTED TRANSITION IDENTIFIABILITY DIAGNOSTIC")
print("=" * 104)

print("\nOracle opened for diagnostics")
print("-" * 104)

print(
    "true pair-law coefficients    : "
    f"{oracle_pair_law}"
)

print(
    f"true persistent triads         : "
    f"{len(oracle_triads)}"
)

print(
    "oracle used for inference      : NO"
)


# =============================================================================
# 1. BUILD TRUE 112-DIMENSIONAL QUOTIENT COEFFICIENT FOR EACH REGIME
#
# Quotient ordering:
#
#     0:28      pair-linear
#     28:56     pair-quadratic
#     56:112    irreducible triad-cross
#
# For pair e:
#
#     theta_e =
#         a_e * [c1, c2]
#
# Persistent true triads occupy only the irreducible C3 coordinate.
# =============================================================================

pair_index = {
    support: q
    for q, support
    in enumerate(TIF_PAIR_SUPPORTS)
}

triad_index = {
    support: q
    for q, support
    in enumerate(TIF_TRIAD_SUPPORTS)
}


def tif_true_pair_amplitudes(stage_index):

    a = np.zeros(
        len(TIF_PAIR_SUPPORTS),
        dtype=float,
    )

    active = oracle_stage_supports[
        stage_index
    ]

    for support in active:

        q = pair_index[
            support
        ]

        a[q] = oracle_weight_map[
            support
        ]

    return a


def tif_true_theta(stage_index):

    a = tif_true_pair_amplitudes(
        stage_index
    )

    theta = np.zeros(
        112,
        dtype=float,
    )

    # pair-linear
    theta[0:28] = (
        oracle_pair_law[0]
        * a
    )

    # pair-quadratic
    theta[28:56] = (
        oracle_pair_law[1]
        * a
    )

    # irreducible triad sector
    for triad in oracle_triads:

        q = triad_index[
            triad
        ]

        theta[
            56 + q
        ] = oracle_triad_strength

    return theta


TIF_ORACLE_THETA = [
    tif_true_theta(m)
    for m in range(
        len(oracle_stage_supports)
    )
]


# =============================================================================
# 2. HELPER: BASELINE PROJECTION IN RAW COORDINATES
#
# Column scaling is used only to determine col(M) robustly.
#
# The projection itself acts on the original raw S and y.
# =============================================================================

def tif_oracle_projector(
    M,
    rel_tol=1.0e-10,
):

    scale_M = np.maximum(
        np.linalg.norm(
            M,
            axis=0,
        ),
        1.0e-14,
    )

    Ms = (
        M
        /
        scale_M[None, :]
    )

    U, s, _ = np.linalg.svd(
        Ms,
        full_matrices=False,
    )

    if s[0] == 0.0:

        rank_M = 0

    else:

        rank_M = int(
            np.sum(
                s > rel_tol * s[0]
            )
        )

    U_M = U[
        :,
        :rank_M
    ]

    return U_M, rank_M


def tif_project_perp(
    A,
    U_M,
):

    if U_M.shape[1] == 0:
        return A.copy()

    return (
        A
        -
        U_M @ (
            U_M.T @ A
        )
    )


# =============================================================================
# 3. RESTRICTED-DESIGN AUDIT
#
# We normalize each column using its PRE-PROJECTION norm.
#
# Therefore small post-projection singular values remain visible as small
# absolute numbers instead of being artificially blown back to O(1).
# =============================================================================

def tif_restricted_audit(
    A_raw,
    A_perp,
):

    scale = np.maximum(
        np.linalg.norm(
            A_raw,
            axis=0,
        ),
        1.0e-14,
    )

    As = (
        A_perp
        /
        scale[None, :]
    )

    s = np.linalg.svd(
        As,
        compute_uv=False,
    )

    ranks = {
        tol: int(
            np.sum(
                s > tol
            )
        )
        for tol in (
            1.0e-6,
            1.0e-8,
            1.0e-10,
            1.0e-12,
        )
    }

    if ranks[1.0e-10] > 0:

        r = ranks[1.0e-10]

        cond_1e10 = (
            s[0]
            /
            s[r - 1]
        )

    else:

        cond_1e10 = np.inf

    visibility = np.linalg.norm(
        A_perp,
        axis=0,
    ) / scale

    return {
        "singular_values": s,
        "ranks": ranks,
        "condition_1e10": cond_1e10,
        "visibility": visibility,
    }


# =============================================================================
# 4. AUDIT EACH TRUE TRANSITION
# =============================================================================

TIF_ORACLE_TRANSITION_AUDIT = []


print("\nTrue transition support")
print("-" * 104)

print(
    "bdry  transition  changed_edges  "
    "||DeltaTheta||  signal_survival  truth_model_err"
)


for k, b in enumerate(
    boundaries,
    start=1,
):

    r = k - 1
    s = k

    # ---------------------------------------------------------------------
    # True absolute and transition coefficients
    # ---------------------------------------------------------------------

    theta_r = TIF_ORACLE_THETA[
        r
    ]

    theta_s = TIF_ORACLE_THETA[
        s
    ]

    delta_theta = (
        theta_s
        -
        theta_r
    )

    a_r = tif_true_pair_amplitudes(
        r
    )

    a_s = tif_true_pair_amplitudes(
        s
    )

    delta_a = (
        a_s - a_r
    )

    changed_edges = np.flatnonzero(
        np.abs(delta_a)
        > 1.0e-14
    )

    # Persistent triads must cancel.
    assert np.max(
        np.abs(
            delta_theta[56:]
        )
    ) < 1.0e-14

    # ---------------------------------------------------------------------
    # Rebuild the BLIND two-sided design from Cell 5.
    # ---------------------------------------------------------------------

    scan = tif_build_two_sided_transition(
        b,
        H,
    )

    M = scan["M"]
    S = scan["S"]
    y = scan["y"]

    U_M, rank_M = (
        tif_oracle_projector(
            M
        )
    )

    S_perp = tif_project_perp(
        S,
        U_M,
    )

    y_perp = tif_project_perp(
        y[:, None],
        U_M,
    ).ravel()

    # ---------------------------------------------------------------------
    # True transition signal before / after nuisance projection
    # ---------------------------------------------------------------------

    signal_raw = (
        S @ delta_theta
    )

    signal_perp = (
        S_perp @ delta_theta
    )

    signal_survival = (
        np.linalg.norm(
            signal_perp
        )
        /
        np.linalg.norm(
            signal_raw
        )
    )

    # ---------------------------------------------------------------------
    # Does the actual synthetic model explain the observed midpoint
    # velocities at the expected numerical accuracy?
    #
    # This is an oracle validation of the observation/library layer,
    # not an inference fit.
    # ---------------------------------------------------------------------

    y_true = (
        M @ theta_r
        +
        S @ delta_theta
    )

    truth_model_error = (
        np.linalg.norm(
            y - y_true
        )
        /
        np.linalg.norm(y)
    )

    projected_truth_error = (
        np.linalg.norm(
            y_perp
            -
            signal_perp
        )
        /
        np.maximum(
            np.linalg.norm(
                y_perp
            ),
            1.0e-14,
        )
    )

    print(
        f"{k:>4d}  "
        f"G{r+1}->G{s+1:<2d}      "
        f"{len(changed_edges):>5d}        "
        f"{np.linalg.norm(delta_theta):.3e}      "
        f"{signal_survival:.3e}       "
        f"{truth_model_error:.3e}"
    )

    # =========================================================================
    # 4A. TRUE CHANGED SUPPORT — UNKNOWN LAW, LIFTED COORDINATES
    #
    # For every changed pair we keep BOTH pair-law coordinates.
    #
    # Number of columns:
    #
    #       2 * number_of_changed_edges.
    #
    # This asks whether the true sparse support is identifiable WITHOUT
    # imposing the shared-law rank-one constraint.
    # =========================================================================

    J_lifted = np.concatenate([
        changed_edges,
        28 + changed_edges,
    ])

    A_lifted_raw = (
        S[:, J_lifted]
    )

    A_lifted_perp = (
        S_perp[:, J_lifted]
    )

    audit_lifted = (
        tif_restricted_audit(
            A_lifted_raw,
            A_lifted_perp,
        )
    )


    # =========================================================================
    # 4B. TRUE CHANGED SUPPORT — SHARED LAW KNOWN
    #
    # Collapse each pair group:
    #
    #       c1 B1_e + c2 B2_e.
    #
    # One unknown scalar Delta a_e per changed edge.
    #
    # This is NOT an inference proposal; it gives an upper-bound diagnostic
    # for what would be identifiable if the law direction were known.
    # =========================================================================

    A_knownlaw_raw = (
        oracle_pair_law[0]
        * S[:, changed_edges]
        +
        oracle_pair_law[1]
        * S[:, 28 + changed_edges]
    )

    A_knownlaw_perp = (
        oracle_pair_law[0]
        * S_perp[:, changed_edges]
        +
        oracle_pair_law[1]
        * S_perp[:, 28 + changed_edges]
    )

    audit_knownlaw = (
        tif_restricted_audit(
            A_knownlaw_raw,
            A_knownlaw_perp,
        )
    )


    # =========================================================================
    # 4C. TRUE SUPPORT — SHARED LAW UNKNOWN BUT COMMON
    #
    # Fix the scale gauge using
    #
    #       c = (1, q)
    #
    # so the true q = 0.5.
    #
    # Unknown local parameters:
    #
    #       Delta a_e  for each changed edge
    #       q          one common interaction-law parameter
    #
    # Jacobian columns:
    #
    # d/d Delta a_e:
    #
    #       S_B1,e + q S_B2,e
    #
    # d/d q:
    #
    #       sum_e Delta a_e S_B2,e.
    #
    # Full local identifiability requires rank:
    #
    #       number_changed_edges + 1.
    # =========================================================================

    q_true = (
        oracle_pair_law[1]
        /
        oracle_pair_law[0]
    )

    edge_jac_raw = (
        S[:, changed_edges]
        +
        q_true
        * S[:, 28 + changed_edges]
    )

    edge_jac_perp = (
        S_perp[:, changed_edges]
        +
        q_true
        * S_perp[:, 28 + changed_edges]
    )

    law_jac_raw = (
        S[:, 28 + changed_edges]
        @ delta_a[
            changed_edges
        ]
    )[:, None]

    law_jac_perp = (
        S_perp[:, 28 + changed_edges]
        @ delta_a[
            changed_edges
        ]
    )[:, None]

    A_sharedlaw_raw = np.hstack([
        edge_jac_raw,
        law_jac_raw,
    ])

    A_sharedlaw_perp = np.hstack([
        edge_jac_perp,
        law_jac_perp,
    ])

    audit_sharedlaw = (
        tif_restricted_audit(
            A_sharedlaw_raw,
            A_sharedlaw_perp,
        )
    )


    TIF_ORACLE_TRANSITION_AUDIT.append({
        "boundary": k,
        "transition": (
            r,
            s,
        ),

        "changed_edges": (
            changed_edges.copy()
        ),

        "changed_edge_supports": [
            TIF_PAIR_SUPPORTS[q]
            for q in changed_edges
        ],

        "delta_a": (
            delta_a.copy()
        ),

        "delta_theta": (
            delta_theta.copy()
        ),

        "signal_survival": (
            signal_survival
        ),

        "truth_model_error": (
            truth_model_error
        ),

        "projected_truth_error": (
            projected_truth_error
        ),

        "lifted_support_audit": (
            audit_lifted
        ),

        "known_law_audit": (
            audit_knownlaw
        ),

        "shared_unknown_law_audit": (
            audit_sharedlaw
        ),
    })


# =============================================================================
# 5. RESTRICTED IDENTIFIABILITY TABLE
# =============================================================================

print("\nRestricted true-support identifiability")
print("-" * 104)

print(
    "bdry  changed  "
    "lifted_dim  lifted_rank@1e-8  "
    "knownlaw_dim  knownlaw_rank@1e-8  "
    "sharedlaw_dim  sharedlaw_rank@1e-8"
)


for result in TIF_ORACLE_TRANSITION_AUDIT:

    n_changed = len(
        result["changed_edges"]
    )

    lifted_dim = (
        2 * n_changed
    )

    knownlaw_dim = (
        n_changed
    )

    sharedlaw_dim = (
        n_changed + 1
    )

    print(
        f"{result['boundary']:>4d}  "
        f"{n_changed:>7d}  "
        f"{lifted_dim:>10d}  "
        f"{result['lifted_support_audit']['ranks'][1.0e-8]:>18d}  "
        f"{knownlaw_dim:>12d}  "
        f"{result['known_law_audit']['ranks'][1.0e-8]:>20d}  "
        f"{sharedlaw_dim:>13d}  "
        f"{result['shared_unknown_law_audit']['ranks'][1.0e-8]:>21d}"
    )


# =============================================================================
# 6. CONDITIONING / VISIBILITY TABLE
# =============================================================================

print("\nTrue-support conditioning")
print("-" * 104)

print(
    "bdry  "
    "signal_survival  "
    "knownlaw_vis_min  "
    "knownlaw_vis_med  "
    "knownlaw_cond@1e-10  "
    "proj_truth_err"
)


for result in TIF_ORACLE_TRANSITION_AUDIT:

    audit = result[
        "known_law_audit"
    ]

    vis = audit[
        "visibility"
    ]

    print(
        f"{result['boundary']:>4d}  "
        f"{result['signal_survival']:.3e}         "
        f"{np.min(vis):.3e}          "
        f"{np.median(vis):.3e}          "
        f"{audit['condition_1e10']:.3e}            "
        f"{result['projected_truth_error']:.3e}"
    )


# =============================================================================
# 7. DISPLAY TRUE CHANGED EDGES
#
# Oracle display only; these values are NOT passed to any estimator.
# =============================================================================

print("\nOracle changed pair supports")
print("-" * 104)

for result in TIF_ORACLE_TRANSITION_AUDIT:

    supports_1b = [
        tuple(
            node + 1
            for node in support
        )
        for support
        in result[
            "changed_edge_supports"
        ]
    ]

    print(
        f"boundary {result['boundary']} : "
        f"{supports_1b}"
    )


# =============================================================================
# 8. SAVE
# =============================================================================

TIF_ORACLE_TRANSITION_DIAGNOSTIC = {
    "audits":
        TIF_ORACLE_TRANSITION_AUDIT,

    "pair_law":
        oracle_pair_law.copy(),

    "oracle_used_for_diagnosis":
        True,

    "oracle_used_for_inference":
        False,
}


print("\n" + "=" * 104)
print("CELL 6 PASSED")
print("=" * 104)

TIF — CELL 6
ORACLE-RESTRICTED TRANSITION IDENTIFIABILITY DIAGNOSTIC

Oracle opened for diagnostics
--------------------------------------------------------------------------------------------------------
true pair-law coefficients    : [1.  0.5]
true persistent triads         : 2
oracle used for inference      : NO

True transition support
--------------------------------------------------------------------------------------------------------
bdry  transition  changed_edges  ||DeltaTheta||  signal_survival  truth_model_err
   1  G1->G2           8        3.216e+00      7.732e-03       9.658e-08
   2  G2->G3           8        3.177e+00      1.514e-03       2.510e-07
   3  G3->G4           6        2.696e+00      3.109e-03       2.954e-07
   4  G4->G5           8        3.163e+00      4.433e-04       1.266e-07
   5  G5->G6           8        3.298e+00      2.607e-04       2.014e-07

Restricted true-support identifiability
----------------------------------------------------------------

In [8]:
# =============================================================================
# TIF STUDY — CELL 7
# TRANSITION IDENTIFIABILITY VS LOCAL OBSERVATION RADIUS
#
# Purpose
# -------
# Cell 5/6 revealed a robust small-neighborhood rank ~ N-1 = 7.
#
# Leading-order theory:
#
#       B(x(t)) -> B(x*)
#
# gives
#
#       rank(P_M^perp S) <= rank(B(x*)) <= N-1.
#
# At finite observation radius H, however,
#
#       B(x(t))
#       =
#       B*
#       + (t-t*) DB* xdot
#       + ...
#
# and higher trajectory jets may provide additional structural directions.
#
# This cell scans the observation radius H and asks:
#
#   1. Does the blind projected transition rank increase with H?
#   2. On the TRUE changed support, does known-law identifiability
#      become full rank?
#   3. How does the weakest true transition singular mode scale with H?
#
# ORACLE IS USED ONLY FOR RESTRICTED IDENTIFIABILITY DIAGNOSTICS.
#
# NO sparse reconstruction.
# =============================================================================

import numpy as np


print("=" * 108)
print("TIF — CELL 7")
print("TRANSITION IDENTIFIABILITY VS LOCAL OBSERVATION RADIUS")
print("=" * 108)


# =============================================================================
# 0. DATA-DERIVED MAXIMUM SAFE RADIUS
#
# Every detected regime has 120 intervals in the present experiment.
#
# For boundary k, H must remain inside the adjacent left/right regimes.
# We derive the common maximum from the data-derived segmentation.
# =============================================================================

segment_counts = np.asarray(
    TIF_OBS["segment_interval_counts"],
    dtype=int,
)

H_MAX = int(
    np.min(segment_counts)
)

assert H_MAX >= 2


# =============================================================================
# 1. RADIUS GRID
#
# Chosen to sample:
#
#   - very local regime
#   - intermediate regime
#   - full adjacent stationary segments
#
# Values are interval counts, not hard-coded physical times.
# =============================================================================

H_candidates_raw = np.array(
    [
        4,
        8,
        12,
        20,
        30,
        40,
        60,
        80,
        100,
        H_MAX,
    ],
    dtype=int,
)

TIF_H_SCAN = np.unique(
    H_candidates_raw[
        H_candidates_raw <= H_MAX
    ]
)

assert len(TIF_H_SCAN) > 0


print("\nRadius scan")
print("-" * 108)

print(
    f"maximum safe radius            : "
    f"{H_MAX} intervals "
    f"({H_MAX * dt:.6f} time units)"
)

print(
    "tested radii                  : "
    + ", ".join(
        f"{H} ({H*dt:.4f})"
        for H in TIF_H_SCAN
    )
)


# =============================================================================
# 2. HELPER:
#    BLIND + TRUE-SUPPORT AUDIT AT ONE BOUNDARY AND ONE H
# =============================================================================

def tif_radius_audit_one(
    boundary_number,
    boundary_index,
    H_local,
):

    # ---------------------------------------------------------------------
    # Rebuild the same information-maximal two-sided design as Cell 5.
    # ---------------------------------------------------------------------

    scan = tif_build_two_sided_transition(
        boundary_index,
        H_local,
    )

    M = scan["M"]
    S = scan["S"]
    y = scan["y"]

    # ---------------------------------------------------------------------
    # Baseline nuisance projection
    # ---------------------------------------------------------------------

    U_M, rank_M = tif_oracle_projector(
        M,
        rel_tol=1.0e-10,
    )

    S_perp = tif_project_perp(
        S,
        U_M,
    )

    y_perp = tif_project_perp(
        y[:, None],
        U_M,
    ).ravel()

    # ---------------------------------------------------------------------
    # BLIND projected transition operator
    #
    # Normalize using PRE-projection S column norms.
    # ---------------------------------------------------------------------

    S_scale = np.maximum(
        np.linalg.norm(
            S,
            axis=0,
        ),
        1.0e-14,
    )

    S_perp_scaled = (
        S_perp
        /
        S_scale[None, :]
    )

    s_blind = np.linalg.svd(
        S_perp_scaled,
        compute_uv=False,
    )

    blind_ranks = {
        tol: int(
            np.sum(
                s_blind > tol
            )
        )
        for tol in (
            1.0e-6,
            1.0e-8,
            1.0e-10,
            1.0e-12,
        )
    }

    # ---------------------------------------------------------------------
    # TRUE transition coefficients
    # ---------------------------------------------------------------------

    r = boundary_number - 1
    s = boundary_number

    theta_r = TIF_ORACLE_THETA[
        r
    ]

    theta_s = TIF_ORACLE_THETA[
        s
    ]

    delta_theta = (
        theta_s - theta_r
    )

    a_r = tif_true_pair_amplitudes(
        r
    )

    a_s = tif_true_pair_amplitudes(
        s
    )

    delta_a = (
        a_s - a_r
    )

    changed_edges = np.flatnonzero(
        np.abs(delta_a)
        > 1.0e-14
    )

    n_changed = len(
        changed_edges
    )

    # ---------------------------------------------------------------------
    # TRUE transition signal survival
    # ---------------------------------------------------------------------

    signal_raw = (
        S @ delta_theta
    )

    signal_perp = (
        S_perp @ delta_theta
    )

    signal_survival = (
        np.linalg.norm(
            signal_perp
        )
        /
        np.maximum(
            np.linalg.norm(
                signal_raw
            ),
            1.0e-14,
        )
    )

    # ---------------------------------------------------------------------
    # TRUE SUPPORT, KNOWN SHARED LAW
    #
    # One scalar Delta a_e per changed edge.
    # ---------------------------------------------------------------------

    A_known_raw = (
        oracle_pair_law[0]
        * S[:, changed_edges]
        +
        oracle_pair_law[1]
        * S[:, 28 + changed_edges]
    )

    A_known_perp = (
        oracle_pair_law[0]
        * S_perp[:, changed_edges]
        +
        oracle_pair_law[1]
        * S_perp[:, 28 + changed_edges]
    )

    known_audit = tif_restricted_audit(
        A_known_raw,
        A_known_perp,
    )

    s_known = known_audit[
        "singular_values"
    ]

    # Since there are exactly n_changed columns, the final singular value
    # measures the weakest direction needed for FULL support identifiability.
    if len(s_known) >= n_changed:

        sigma_min_full = float(
            s_known[
                n_changed - 1
            ]
        )

        sigma_max_full = float(
            s_known[0]
        )

        if sigma_min_full > 0.0:

            cond_full = (
                sigma_max_full
                /
                sigma_min_full
            )

        else:

            cond_full = np.inf

    else:

        sigma_min_full = 0.0
        cond_full = np.inf

    # ---------------------------------------------------------------------
    # TRUE SUPPORT, SHARED LAW UNKNOWN BUT COMMON
    #
    # Gauge:
    #
    #       c = (1, q)
    #
    # parameters:
    #
    #       Delta a_e for each changed edge
    #       q
    # ---------------------------------------------------------------------

    q_true = (
        oracle_pair_law[1]
        /
        oracle_pair_law[0]
    )

    edge_jac_raw = (
        S[:, changed_edges]
        +
        q_true
        * S[:, 28 + changed_edges]
    )

    edge_jac_perp = (
        S_perp[:, changed_edges]
        +
        q_true
        * S_perp[:, 28 + changed_edges]
    )

    law_jac_raw = (
        S[:, 28 + changed_edges]
        @ delta_a[
            changed_edges
        ]
    )[:, None]

    law_jac_perp = (
        S_perp[:, 28 + changed_edges]
        @ delta_a[
            changed_edges
        ]
    )[:, None]

    A_shared_raw = np.hstack(
        [
            edge_jac_raw,
            law_jac_raw,
        ]
    )

    A_shared_perp = np.hstack(
        [
            edge_jac_perp,
            law_jac_perp,
        ]
    )

    shared_audit = tif_restricted_audit(
        A_shared_raw,
        A_shared_perp,
    )

    # ---------------------------------------------------------------------
    # Oracle model consistency at this radius
    # ---------------------------------------------------------------------

    y_true = (
        M @ theta_r
        +
        S @ delta_theta
    )

    truth_error = (
        np.linalg.norm(
            y - y_true
        )
        /
        np.linalg.norm(y)
    )

    # ---------------------------------------------------------------------
    # How accurately does the true projected transition reproduce the
    # baseline-eliminated observable?
    # ---------------------------------------------------------------------

    projected_truth_error = (
        np.linalg.norm(
            y_perp
            -
            signal_perp
        )
        /
        np.maximum(
            np.linalg.norm(
                y_perp
            ),
            1.0e-14,
        )
    )

    return {
        "boundary_number":
            boundary_number,

        "boundary_time":
            t_state[
                boundary_index
            ],

        "H_steps":
            int(H_local),

        "H_time":
            float(
                H_local * dt
            ),

        "n_changed":
            n_changed,

        "changed_edges":
            changed_edges.copy(),

        "rank_M":
            rank_M,

        "blind_ranks":
            blind_ranks,

        "blind_singular_values":
            s_blind,

        "signal_survival":
            signal_survival,

        "knownlaw_rank_1e6":
            known_audit[
                "ranks"
            ][1.0e-6],

        "knownlaw_rank_1e8":
            known_audit[
                "ranks"
            ][1.0e-8],

        "knownlaw_rank_1e10":
            known_audit[
                "ranks"
            ][1.0e-10],

        "knownlaw_sigma_min_full":
            sigma_min_full,

        "knownlaw_condition_full":
            cond_full,

        "knownlaw_visibility":
            known_audit[
                "visibility"
            ],

        "sharedlaw_dim":
            n_changed + 1,

        "sharedlaw_rank_1e8":
            shared_audit[
                "ranks"
            ][1.0e-8],

        "sharedlaw_rank_1e10":
            shared_audit[
                "ranks"
            ][1.0e-10],

        "truth_model_error":
            truth_error,

        "projected_truth_error":
            projected_truth_error,
    }


# =============================================================================
# 3. RUN RADIUS SCAN
# =============================================================================

TIF_RADIUS_SCAN_RESULTS = []


for boundary_number, boundary_index in enumerate(
    boundaries,
    start=1,
):

    for H_local in TIF_H_SCAN:

        result = tif_radius_audit_one(
            boundary_number,
            boundary_index,
            int(H_local),
        )

        TIF_RADIUS_SCAN_RESULTS.append(
            result
        )


# =============================================================================
# 4. REPORT ONE TABLE PER TRANSITION
# =============================================================================

for boundary_number in range(
    1,
    len(boundaries) + 1,
):

    rows = [
        r
        for r in TIF_RADIUS_SCAN_RESULTS
        if (
            r["boundary_number"]
            ==
            boundary_number
        )
    ]

    n_changed = rows[0][
        "n_changed"
    ]

    print(
        "\n"
        + "=" * 108
    )

    print(
        f"Boundary {boundary_number}: "
        f"G{boundary_number}->G{boundary_number + 1} "
        f"| true changed edges = {n_changed}"
    )

    print(
        "-" * 108
    )

    print(
        "H_time   blind_rank@1e-8  "
        "known_rank@1e-8  known_rank@1e-10  "
        "sigma_min(full)  cond(full)    signal_survival"
    )

    for r in rows:

        print(
            f"{r['H_time']:.4f}   "
            f"{r['blind_ranks'][1.0e-8]:>15d}  "
            f"{r['knownlaw_rank_1e8']:>15d}  "
            f"{r['knownlaw_rank_1e10']:>16d}  "
            f"{r['knownlaw_sigma_min_full']:.3e}       "
            f"{r['knownlaw_condition_full']:.3e}   "
            f"{r['signal_survival']:.3e}"
        )


# =============================================================================
# 5. SUMMARY:
#    FIRST RADIUS AT WHICH TRUE KNOWN-LAW SUPPORT BECOMES FULL RANK
# =============================================================================

print("\n" + "=" * 108)
print("FULL TRUE-SUPPORT IDENTIFIABILITY SUMMARY")
print("-" * 108)

print(
    "bdry  changed_edges  first_full_H@1e-8  "
    "first_full_H@1e-10  best_sigma_min"
)


for boundary_number in range(
    1,
    len(boundaries) + 1,
):

    rows = [
        r
        for r in TIF_RADIUS_SCAN_RESULTS
        if (
            r["boundary_number"]
            ==
            boundary_number
        )
    ]

    n_changed = rows[0][
        "n_changed"
    ]

    full_1e8 = [
        r
        for r in rows
        if (
            r["knownlaw_rank_1e8"]
            ==
            n_changed
        )
    ]

    full_1e10 = [
        r
        for r in rows
        if (
            r["knownlaw_rank_1e10"]
            ==
            n_changed
        )
    ]

    if len(full_1e8) > 0:

        first_h_1e8 = (
            full_1e8[0][
                "H_time"
            ]
        )

        first_h_1e8_str = (
            f"{first_h_1e8:.4f}"
        )

    else:

        first_h_1e8_str = (
            "NONE"
        )

    if len(full_1e10) > 0:

        first_h_1e10 = (
            full_1e10[0][
                "H_time"
            ]
        )

        first_h_1e10_str = (
            f"{first_h_1e10:.4f}"
        )

    else:

        first_h_1e10_str = (
            "NONE"
        )

    best_sigma = max(
        r[
            "knownlaw_sigma_min_full"
        ]
        for r in rows
    )

    print(
        f"{boundary_number:>4d}  "
        f"{n_changed:>13d}  "
        f"{first_h_1e8_str:>17s}  "
        f"{first_h_1e10_str:>18s}  "
        f"{best_sigma:.3e}"
    )


# =============================================================================
# 6. SHARED-UNKNOWN-LAW SUMMARY
# =============================================================================

print("\nShared-unknown-law local rank at maximum radius")
print("-" * 108)

print(
    "bdry  changed_edges  parameter_dim  "
    "rank@1e-8  rank@1e-10"
)


for boundary_number in range(
    1,
    len(boundaries) + 1,
):

    rows = [
        r
        for r in TIF_RADIUS_SCAN_RESULTS
        if (
            r["boundary_number"]
            ==
            boundary_number
        )
    ]

    rmax = rows[-1]

    print(
        f"{boundary_number:>4d}  "
        f"{rmax['n_changed']:>13d}  "
        f"{rmax['sharedlaw_dim']:>13d}  "
        f"{rmax['sharedlaw_rank_1e8']:>9d}  "
        f"{rmax['sharedlaw_rank_1e10']:>10d}"
    )


# =============================================================================
# 7. NUMERICAL CONSISTENCY
# =============================================================================

max_truth_error = max(
    r["truth_model_error"]
    for r in TIF_RADIUS_SCAN_RESULTS
)

max_projected_truth_error = max(
    r["projected_truth_error"]
    for r in TIF_RADIUS_SCAN_RESULTS
)


print("\nNumerical consistency")
print("-" * 108)

print(
    f"max true-model relative error  : "
    f"{max_truth_error:.3e}"
)

print(
    f"max projected truth error      : "
    f"{max_projected_truth_error:.3e}"
)


# =============================================================================
# 8. SAVE
# =============================================================================

TIF_RADIUS_IDENTIFIABILITY = {
    "H_scan_steps":
        TIF_H_SCAN.copy(),

    "H_scan_time":
        TIF_H_SCAN.astype(float)
        * dt,

    "results":
        TIF_RADIUS_SCAN_RESULTS,

    "oracle_used_for_diagnosis":
        True,

    "oracle_used_for_inference":
        False,
}


print("\n" + "=" * 108)
print("CELL 7 PASSED")
print("=" * 108)

TIF — CELL 7
TRANSITION IDENTIFIABILITY VS LOCAL OBSERVATION RADIUS

Radius scan
------------------------------------------------------------------------------------------------------------
maximum safe radius            : 120 intervals (0.060000 time units)
tested radii                  : 4 (0.0020), 8 (0.0040), 12 (0.0060), 20 (0.0100), 30 (0.0150), 40 (0.0200), 60 (0.0300), 80 (0.0400), 100 (0.0500), 120 (0.0600)

Boundary 1: G1->G2 | true changed edges = 8
------------------------------------------------------------------------------------------------------------
H_time   blind_rank@1e-8  known_rank@1e-8  known_rank@1e-10  sigma_min(full)  cond(full)    signal_survival
0.0020                 7                7                 7  2.028e-11       1.734e+08   1.145e-03
0.0040                 7                7                 7  3.939e-12       6.181e+08   8.914e-04
0.0060                 7                7                 7  2.467e-12       2.993e+09   2.276e-03
0.0100               

In [9]:
# =============================================================================
# TIF STUDY — CELL 8
# GLOBAL CUMULATIVE-TRANSITION IDENTIFIABILITY
#
# Purpose
# -------
# Cells 5--7 treated every transition locally:
#
#       theta_r = nuisance baseline
#       Delta_theta_r = local transition
#
# and found a robust small-neighborhood ceiling ~ N-1 = 7.
#
# But globally:
#
#       theta_m
#       =
#       theta_1
#       + Delta_theta_1
#       + ...
#       + Delta_theta_{m-1}.
#
# Therefore there is only ONE absolute baseline theta_1.
#
# The full trajectory satisfies
#
#       v(t)
#       =
#       B(x(t)) theta_1
#       +
#       sum_k I_k(t) B(x(t)) Delta_theta_k,
#
# where I_k(t)=1 after transition k and 0 before it.
#
# This cell tests whether cumulative temporal structure restores
# identifiability of the TRUE transition sequence.
#
# ORACLE:
#   used ONLY to specify true changed supports and true coefficients
#   for identifiability diagnostics.
#
# NO support inference.
# NO sparse solver.
# =============================================================================

import numpy as np


print("=" * 112)
print("TIF — CELL 8")
print("GLOBAL CUMULATIVE-TRANSITION IDENTIFIABILITY")
print("=" * 112)


# =============================================================================
# 0. GLOBAL OBSERVABLE DESIGN
# =============================================================================

Q = 112

M_global = (
    Bq_mid
    .reshape(-1, Q)
)

y_global = (
    V
    .reshape(-1)
)

n_intervals = len(V)


# Data-derived segment label for every trajectory interval.
stage_of_interval = np.empty(
    n_intervals,
    dtype=int,
)

for stage, sl in enumerate(
    TIF_OBS["segment_slices"]
):
    stage_of_interval[sl] = stage


assert np.all(
    stage_of_interval >= 0
)


print("\nGlobal geometry")
print("-" * 112)

print(
    f"trajectory intervals           : "
    f"{n_intervals}"
)

print(
    f"scalar observation rows        : "
    f"{len(y_global)}"
)

print(
    f"absolute baseline dimension    : "
    f"{Q}"
)

print(
    f"number of transitions          : "
    f"{len(boundaries)}"
)


# =============================================================================
# 1. GLOBAL BASELINE NUISANCE PROJECTION
#
# Only theta_1 is treated as the absolute nuisance parameter.
# =============================================================================

U_global, rank_M_global = (
    tif_oracle_projector(
        M_global,
        rel_tol=1.0e-10,
    )
)

y_global_perp = tif_project_perp(
    y_global[:, None],
    U_global,
).ravel()


print(
    f"global baseline rank           : "
    f"{rank_M_global}"
)


# =============================================================================
# 2. TRUE TRANSITION SUPPORTS AND AMPLITUDES
#
# Build:
#
#       Delta a_1, ..., Delta a_5.
#
# A transition k contributes to every later regime.
# =============================================================================

transition_changed_edges = []
transition_delta_a = []

for k in range(
    len(boundaries)
):

    a_before = tif_true_pair_amplitudes(
        k
    )

    a_after = tif_true_pair_amplitudes(
        k + 1
    )

    delta_a = (
        a_after - a_before
    )

    changed = np.flatnonzero(
        np.abs(delta_a)
        > 1.0e-14
    )

    transition_changed_edges.append(
        changed
    )

    transition_delta_a.append(
        delta_a
    )


n_changed_each = [
    len(x)
    for x in transition_changed_edges
]

total_changed_parameters = sum(
    n_changed_each
)


print("\nTrue transition parameterization")
print("-" * 112)

print(
    "changed edges / transition    : "
    + ", ".join(
        str(n)
        for n in n_changed_each
    )
)

print(
    f"total Delta-a parameters       : "
    f"{total_changed_parameters}"
)


# =============================================================================
# 3. BUILD GLOBAL KNOWN-LAW TRANSITION DESIGN
#
# For transition k:
#
#       column(e,k)
#       =
#       I(stage > k)
#       * [ c1 B1_e(x) + c2 B2_e(x) ].
#
# Each transition-edge pair is its own parameter, even if the same edge
# changes at more than one transition.
# =============================================================================

pair_field_true = (
    oracle_pair_law[0]
    * TIF_P1
    +
    oracle_pair_law[1]
    * TIF_P2
)


known_columns = []
known_labels = []
true_delta_vector = []


for k, changed in enumerate(
    transition_changed_edges
):

    active_intervals = (
        stage_of_interval
        >= (k + 1)
    )

    for e in changed:

        col_tensor = np.zeros(
            (
                n_intervals,
                N,
            ),
            dtype=float,
        )

        col_tensor[
            active_intervals
        ] = pair_field_true[
            active_intervals,
            :,
            e,
        ]

        known_columns.append(
            col_tensor.reshape(-1)
        )

        known_labels.append(
            (
                k,
                int(e),
            )
        )

        true_delta_vector.append(
            transition_delta_a[k][e]
        )


S_known_global = np.column_stack(
    known_columns
)

true_delta_vector = np.asarray(
    true_delta_vector,
    dtype=float,
)


assert S_known_global.shape == (
    len(y_global),
    total_changed_parameters,
)


# =============================================================================
# 4. PROJECT KNOWN-LAW TRANSITIONS AGAINST THE SINGLE GLOBAL BASELINE
# =============================================================================

S_known_perp = tif_project_perp(
    S_known_global,
    U_global,
)

known_global_audit = tif_restricted_audit(
    S_known_global,
    S_known_perp,
)


# =============================================================================
# 5. ORACLE MODEL CONSISTENCY
#
# Full cumulative transition model:
#
#       y
#       =
#       M theta_1
#       +
#       S Delta-a.
#
# =============================================================================

theta_1_true = (
    TIF_ORACLE_THETA[0]
)

y_true_global = (
    M_global @ theta_1_true
    +
    S_known_global @ true_delta_vector
)

truth_global_error = (
    np.linalg.norm(
        y_global
        -
        y_true_global
    )
    /
    np.linalg.norm(
        y_global
    )
)


signal_raw = (
    S_known_global
    @ true_delta_vector
)

signal_perp = (
    S_known_perp
    @ true_delta_vector
)

global_signal_survival = (
    np.linalg.norm(
        signal_perp
    )
    /
    np.linalg.norm(
        signal_raw
    )
)

projected_truth_error = (
    np.linalg.norm(
        y_global_perp
        -
        signal_perp
    )
    /
    np.maximum(
        np.linalg.norm(
            y_global_perp
        ),
        1.0e-14,
    )
)


# =============================================================================
# 6. UNRESTRICTED LIFTED TRUE-SUPPORT DESIGN
#
# Unknown law coordinates remain independent:
#
#       Delta theta_e =
#       (Delta theta_e,linear,
#        Delta theta_e,quadratic).
#
# Dimension:
#
#       2 * total_changed_parameters.
#
# This checks how much of the gain comes purely from cumulative temporal
# structure before imposing the stationary-law direction.
# =============================================================================

lifted_columns = []


for k, changed in enumerate(
    transition_changed_edges
):

    active_intervals = (
        stage_of_interval
        >= (k + 1)
    )

    # linear coordinates
    for e in changed:

        col = np.zeros(
            (
                n_intervals,
                N,
            ),
            dtype=float,
        )

        col[
            active_intervals
        ] = TIF_P1[
            active_intervals,
            :,
            e,
        ]

        lifted_columns.append(
            col.reshape(-1)
        )

    # quadratic coordinates
    for e in changed:

        col = np.zeros(
            (
                n_intervals,
                N,
            ),
            dtype=float,
        )

        col[
            active_intervals
        ] = TIF_P2[
            active_intervals,
            :,
            e,
        ]

        lifted_columns.append(
            col.reshape(-1)
        )


S_lifted_global = np.column_stack(
    lifted_columns
)

S_lifted_perp = tif_project_perp(
    S_lifted_global,
    U_global,
)

lifted_global_audit = tif_restricted_audit(
    S_lifted_global,
    S_lifted_perp,
)


# =============================================================================
# 7. SHARED UNKNOWN LAW — GLOBAL LOCAL IDENTIFIABILITY
#
# Gauge:
#
#       c = (1, q)
#
# with one GLOBAL q shared by all five transitions.
#
# Parameters:
#
#       38 transition amplitudes
#       + 1 shared q
#
#       = 39 total parameters.
#
# Jacobian w.r.t. Delta a:
#
#       I_k(t) [B1_e + q B2_e].
#
# Jacobian w.r.t. q:
#
#       sum_{k,e}
#       I_k(t) Delta a_{k,e} B2_e.
# =============================================================================

q_true = (
    oracle_pair_law[1]
    /
    oracle_pair_law[0]
)


shared_edge_columns = []
law_column_tensor = np.zeros(
    (
        n_intervals,
        N,
    ),
    dtype=float,
)


for k, changed in enumerate(
    transition_changed_edges
):

    active_intervals = (
        stage_of_interval
        >= (k + 1)
    )

    for e in changed:

        # d / d Delta-a_(k,e)
        col = np.zeros(
            (
                n_intervals,
                N,
            ),
            dtype=float,
        )

        col[
            active_intervals
        ] = (
            TIF_P1[
                active_intervals,
                :,
                e,
            ]
            +
            q_true
            * TIF_P2[
                active_intervals,
                :,
                e,
            ]
        )

        shared_edge_columns.append(
            col.reshape(-1)
        )

        # contribution to d / dq
        law_column_tensor[
            active_intervals
        ] += (
            transition_delta_a[k][e]
            *
            TIF_P2[
                active_intervals,
                :,
                e,
            ]
        )


law_column = (
    law_column_tensor
    .reshape(-1, 1)
)


S_shared_global = np.column_stack(
    shared_edge_columns
)

J_shared_global = np.hstack(
    [
        S_shared_global,
        law_column,
    ]
)

J_shared_perp = tif_project_perp(
    J_shared_global,
    U_global,
)

shared_global_audit = tif_restricted_audit(
    J_shared_global,
    J_shared_perp,
)


# =============================================================================
# 8. JOINT RANK WITH BASELINE
#
# Direct check:
#
#       rank([M,S]) - rank(M)
#
# using independently normalized column blocks.
# =============================================================================

M_scale = np.maximum(
    np.linalg.norm(
        M_global,
        axis=0,
    ),
    1.0e-14,
)

S_scale = np.maximum(
    np.linalg.norm(
        S_known_global,
        axis=0,
    ),
    1.0e-14,
)

M_scaled = (
    M_global
    /
    M_scale[None, :]
)

S_scaled = (
    S_known_global
    /
    S_scale[None, :]
)

joint_known = np.hstack(
    [
        M_scaled,
        S_scaled,
    ]
)

rank_joint_known, _ = (
    tif_rank_from_scaled_matrix(
        joint_known,
        rel_tol=1.0e-10,
    )
)

rank_gain_known = (
    rank_joint_known
    -
    rank_M_global
)


# =============================================================================
# 9. REPORT
# =============================================================================

print("\nGlobal cumulative identifiability")
print("-" * 112)

print(
    "model                               dim     "
    "rank@1e-6  rank@1e-8  rank@1e-10  rank@1e-12"
)

print(
    f"lifted transitions                  "
    f"{S_lifted_global.shape[1]:>3d}     "
    f"{lifted_global_audit['ranks'][1.0e-6]:>9d}  "
    f"{lifted_global_audit['ranks'][1.0e-8]:>9d}  "
    f"{lifted_global_audit['ranks'][1.0e-10]:>10d}  "
    f"{lifted_global_audit['ranks'][1.0e-12]:>10d}"
)

print(
    f"known shared law                    "
    f"{S_known_global.shape[1]:>3d}     "
    f"{known_global_audit['ranks'][1.0e-6]:>9d}  "
    f"{known_global_audit['ranks'][1.0e-8]:>9d}  "
    f"{known_global_audit['ranks'][1.0e-10]:>10d}  "
    f"{known_global_audit['ranks'][1.0e-12]:>10d}"
)

print(
    f"shared unknown law (local Jacobian) "
    f"{J_shared_global.shape[1]:>3d}     "
    f"{shared_global_audit['ranks'][1.0e-6]:>9d}  "
    f"{shared_global_audit['ranks'][1.0e-8]:>9d}  "
    f"{shared_global_audit['ranks'][1.0e-10]:>10d}  "
    f"{shared_global_audit['ranks'][1.0e-12]:>10d}"
)


print("\nKnown-law transition conditioning")
print("-" * 112)

s_known = (
    known_global_audit[
        "singular_values"
    ]
)

print(
    f"parameter dimension             : "
    f"{total_changed_parameters}"
)

print(
    f"rank([M,S]) - rank(M) @1e-10   : "
    f"{rank_gain_known}"
)

print(
    f"smallest transition singular    : "
    f"{s_known[-1]:.6e}"
)

print(
    f"largest transition singular     : "
    f"{s_known[0]:.6e}"
)

print(
    f"full condition number           : "
    f"{s_known[0] / s_known[-1]:.6e}"
)

print(
    f"median column visibility        : "
    f"{np.median(known_global_audit['visibility']):.6e}"
)

print(
    f"minimum column visibility       : "
    f"{np.min(known_global_audit['visibility']):.6e}"
)


print("\nOracle consistency")
print("-" * 112)

print(
    f"true cumulative-model error     : "
    f"{truth_global_error:.6e}"
)

print(
    f"transition signal survival      : "
    f"{global_signal_survival:.6e}"
)

print(
    f"projected true-signal error     : "
    f"{projected_truth_error:.6e}"
)


# =============================================================================
# 10. PER-TRANSITION BLOCK VISIBILITY
#
# Even though inference is global, inspect whether any transition block
# remains systematically weak.
# =============================================================================

print("\nKnown-law visibility by transition")
print("-" * 112)

print(
    "transition  changed_edges  "
    "vis_min      vis_median    vis_max"
)


offset = 0

for k, n_changed in enumerate(
    n_changed_each,
    start=1,
):

    vis_block = known_global_audit[
        "visibility"
    ][
        offset:
        offset + n_changed
    ]

    print(
        f"G{k}->G{k+1:<2d}    "
        f"{n_changed:>6d}       "
        f"{np.min(vis_block):.3e}    "
        f"{np.median(vis_block):.3e}    "
        f"{np.max(vis_block):.3e}"
    )

    offset += n_changed


# =============================================================================
# 11. SAVE
# =============================================================================

TIF_GLOBAL_TRANSITION_IDENTIFIABILITY = {
    "rank_baseline":
        rank_M_global,

    "n_changed_each":
        np.asarray(
            n_changed_each,
            dtype=int,
        ),

    "known_transition_design":
        S_known_global,

    "known_transition_projected":
        S_known_perp,

    "known_law_audit":
        known_global_audit,

    "lifted_audit":
        lifted_global_audit,

    "shared_unknown_law_audit":
        shared_global_audit,

    "truth_model_error":
        truth_global_error,

    "signal_survival":
        global_signal_survival,

    "oracle_used_for_diagnosis":
        True,

    "oracle_used_for_inference":
        False,
}


print("\n" + "=" * 112)
print("CELL 8 PASSED")
print("=" * 112)

TIF — CELL 8
GLOBAL CUMULATIVE-TRANSITION IDENTIFIABILITY

Global geometry
----------------------------------------------------------------------------------------------------------------
trajectory intervals           : 720
scalar observation rows        : 5760
absolute baseline dimension    : 112
number of transitions          : 5
global baseline rank           : 111

True transition parameterization
----------------------------------------------------------------------------------------------------------------
changed edges / transition    : 8, 8, 6, 8, 8
total Delta-a parameters       : 38

Global cumulative identifiability
----------------------------------------------------------------------------------------------------------------
model                               dim     rank@1e-6  rank@1e-8  rank@1e-10  rank@1e-12
lifted transitions                   76            50         62          70          74
known shared law                     38            38         38         

In [10]:
# =============================================================================
# TIF STUDY — CELL 9
# PROFILED SUPPORT-SEPARATION AUDIT
#
# Purpose
# -------
# Test whether the GLOBAL cumulative transition model can distinguish
# the true transition support from nearby / wrong supports.
#
# Core profiled score:
#
#       R(J)
#       =
#       min_{||c||=1, delta}
#       || y_perp - S_J(c) delta ||_2
#
# where
#
#       y_perp = P_M^perp y
#
# and M is the single global absolute-baseline design.
#
# For L=2:
#
#       c(theta) = (cos theta, sin theta),
#       theta in [0, pi).
#
# Fixed theta -> ordinary linear least squares in transition amplitudes.
#
# This cell uses the oracle ONLY to define diagnostic support sets:
#
#   1. true support J*
#   2. all leave-one-out supports J* \ {q}
#   3. random one-swap supports
#   4. random same-cardinality wrong supports
#
# NO support-search algorithm is run.
# =============================================================================

import numpy as np
from scipy.optimize import minimize_scalar


print("=" * 112)
print("TIF — CELL 9")
print("PROFILED SUPPORT-SEPARATION AUDIT")
print("=" * 112)


# =============================================================================
# 0. GLOBAL PROJECTED TARGET
# =============================================================================

y_perp = np.asarray(
    y_global_perp,
    dtype=float,
)

n_rows = len(y_perp)

y2 = float(
    y_perp @ y_perp
)

y_norm = np.sqrt(
    y2
)


print("\nProjected target")
print("-" * 112)

print(
    f"scalar rows                    : "
    f"{n_rows}"
)

print(
    f"||y_perp||_2                  : "
    f"{y_norm:.6e}"
)


# =============================================================================
# 1. BUILD ALL 140 CANDIDATE TRANSITION-EVENT COLUMNS
#
# Candidate event:
#
#       q = (transition k, pair edge e)
#
# There are:
#
#       5 transitions * 28 possible pair supports = 140 events.
#
# For each event we keep the TWO weak pair-law coordinates separately:
#
#       S1_q : pair-linear contribution
#       S2_q : pair-quadratic contribution
#
# A shared law direction c(theta) combines them as
#
#       s_q(theta)
#       =
#       cos(theta) S1_q
#       +
#       sin(theta) S2_q.
# =============================================================================

n_transitions = len(
    boundaries
)

n_pairs = len(
    TIF_PAIR_SUPPORTS
)

n_events = (
    n_transitions
    * n_pairs
)


S1_columns = []
S2_columns = []
event_labels = []


for k in range(
    n_transitions
):

    active = (
        stage_of_interval
        >= (k + 1)
    )

    for e in range(
        n_pairs
    ):

        col1 = np.zeros(
            (
                n_intervals,
                N,
            ),
            dtype=float,
        )

        col2 = np.zeros_like(
            col1
        )

        col1[
            active
        ] = TIF_P1[
            active,
            :,
            e,
        ]

        col2[
            active
        ] = TIF_P2[
            active,
            :,
            e,
        ]

        S1_columns.append(
            col1.reshape(-1)
        )

        S2_columns.append(
            col2.reshape(-1)
        )

        event_labels.append(
            (
                k,
                e,
            )
        )


S1_all = np.column_stack(
    S1_columns
)

S2_all = np.column_stack(
    S2_columns
)


assert S1_all.shape == (
    len(y_global),
    n_events,
)

assert S2_all.shape == (
    len(y_global),
    n_events,
)


# =============================================================================
# 2. PROJECT OUT THE SINGLE GLOBAL BASELINE
# =============================================================================

S1_perp = tif_project_perp(
    S1_all,
    U_global,
)

S2_perp = tif_project_perp(
    S2_all,
    U_global,
)


# =============================================================================
# 3. PRECOMPUTE ALL INNER PRODUCTS
#
# For fixed theta:
#
#       S(theta)
#       =
#       c1 S1 + c2 S2.
#
# For support J:
#
#       G_J(theta) = S_J(theta)^T S_J(theta)
#       g_J(theta) = S_J(theta)^T y_perp.
#
# We can therefore evaluate the profile without repeatedly touching
# all 5760 trajectory rows.
# =============================================================================

G11 = (
    S1_perp.T
    @ S1_perp
)

G12 = (
    S1_perp.T
    @ S2_perp
)

G22 = (
    S2_perp.T
    @ S2_perp
)

g1 = (
    S1_perp.T
    @ y_perp
)

g2 = (
    S2_perp.T
    @ y_perp
)


# =============================================================================
# 4. SUPPORT / EVENT HELPERS
# =============================================================================

def tif_event_id(
    transition_index,
    pair_index,
):
    return (
        transition_index
        * n_pairs
        +
        pair_index
    )


def tif_event_description(
    event_id,
):

    k, e = event_labels[
        event_id
    ]

    pair = TIF_PAIR_SUPPORTS[
        e
    ]

    pair_1b = tuple(
        node + 1
        for node in pair
    )

    return (
        f"G{k+1}->G{k+2}, "
        f"edge={pair_1b}"
    )


# =============================================================================
# 5. TRUE GLOBAL SUPPORT — ORACLE DIAGNOSTIC ONLY
# =============================================================================

J_true = []


for k, changed in enumerate(
    transition_changed_edges
):

    for e in changed:

        J_true.append(
            tif_event_id(
                k,
                int(e),
            )
        )


J_true = tuple(
    sorted(
        J_true
    )
)

J_true_set = set(
    J_true
)

assert len(
    J_true
) == total_changed_parameters


all_events = np.arange(
    n_events,
    dtype=int,
)

false_events = np.array(
    [
        q
        for q in all_events
        if q not in J_true_set
    ],
    dtype=int,
)


print("\nCandidate-event space")
print("-" * 112)

print(
    f"possible transition-edge events: "
    f"{n_events}"
)

print(
    f"true changed events             : "
    f"{len(J_true)}"
)

print(
    f"false candidate events          : "
    f"{len(false_events)}"
)


# =============================================================================
# 6. FIXED-THETA LEAST-SQUARE SCORE
#
# Uses the Gram representation.
#
# For fixed theta and support J:
#
#       delta_hat = argmin ||y - S_J(theta) delta||.
#
# We solve the normal equations with an eigenvalue pseudoinverse.
#
# This is appropriate for this diagnostic because the support dimensions
# are at most ~38 and all trajectory Gram products are precomputed.
# =============================================================================

def tif_fixed_theta_score(
    J,
    theta,
    eig_rtol=1.0e-12,
):

    J = np.asarray(
        J,
        dtype=int,
    )

    if len(J) == 0:

        return {
            "rss": y2,
            "relative": 1.0,
            "rms": (
                y_norm
                /
                np.sqrt(n_rows)
            ),
            "delta": np.zeros(
                0,
                dtype=float,
            ),
            "rank": 0,
        }

    theta = float(
        theta % np.pi
    )

    c1 = np.cos(
        theta
    )

    c2 = np.sin(
        theta
    )

    G = (
        (c1 * c1)
        * G11[
            np.ix_(J, J)
        ]
        +
        (c1 * c2)
        * (
            G12[
                np.ix_(J, J)
            ]
            +
            G12.T[
                np.ix_(J, J)
            ]
        )
        +
        (c2 * c2)
        * G22[
            np.ix_(J, J)
        ]
    )

    g = (
        c1
        * g1[J]
        +
        c2
        * g2[J]
    )

    # Symmetrize against numerical roundoff.
    G = 0.5 * (
        G + G.T
    )

    evals, evecs = np.linalg.eigh(
        G
    )

    lam_max = float(
        np.max(
            evals
        )
    )

    if lam_max <= 0.0:

        delta = np.zeros(
            len(J),
            dtype=float,
        )

        rank = 0

    else:

        keep = (
            evals
            >
            eig_rtol
            * lam_max
        )

        rank = int(
            np.sum(
                keep
            )
        )

        if rank == 0:

            delta = np.zeros(
                len(J),
                dtype=float,
            )

        else:

            V = evecs[
                :,
                keep
            ]

            lam = evals[
                keep
            ]

            delta = (
                V
                @ (
                    (
                        V.T @ g
                    )
                    /
                    lam
                )
            )

    # More stable than y2 - g^T delta alone.
    rss = (
        y2
        -
        2.0
        * float(
            delta @ g
        )
        +
        float(
            delta @ (
                G @ delta
            )
        )
    )

    # Protect against tiny negative roundoff.
    rss = max(
        rss,
        0.0,
    )

    residual_norm = np.sqrt(
        rss
    )

    return {
        "rss": rss,

        "relative": (
            residual_norm
            /
            y_norm
        ),

        "rms": (
            residual_norm
            /
            np.sqrt(
                n_rows
            )
        ),

        "delta": delta,

        "rank": rank,
    }


# =============================================================================
# 7. PROFILE OVER THE SHARED LAW DIRECTION
#
# L=2 -> projective circle:
#
#       theta in [0, pi).
#
# Strategy:
#
#   1. deterministic coarse scan every 2 degrees;
#   2. refine around the best coarse direction with bounded scalar search.
#
# depth =
#
#       median(profile residual)
#       --------------------------------
#       best profile residual
#
# with a numerical denominator floor.
# =============================================================================

theta_grid = np.linspace(
    0.0,
    np.pi,
    90,
    endpoint=False,
)


def tif_profile_support(
    J,
):

    J = tuple(
        sorted(
            set(
                int(q)
                for q in J
            )
        )
    )

    coarse_rel = np.empty(
        len(theta_grid),
        dtype=float,
    )

    for i, theta in enumerate(
        theta_grid
    ):

        coarse_rel[i] = (
            tif_fixed_theta_score(
                J,
                theta,
            )["relative"]
        )

    i_best = int(
        np.argmin(
            coarse_rel
        )
    )

    theta0 = float(
        theta_grid[
            i_best
        ]
    )

    # Coarse spacing.
    dtheta = (
        np.pi
        /
        len(theta_grid)
    )

    # Periodic objective.
    def objective(theta):

        return tif_fixed_theta_score(
            J,
            theta % np.pi,
        )["relative"]

    # Search locally around the best coarse point.
    opt = minimize_scalar(
        objective,
        bounds=(
            theta0 - dtheta,
            theta0 + dtheta,
        ),
        method="bounded",
        options={
            "xatol": 1.0e-10,
            "maxiter": 80,
        },
    )

    theta_best = float(
        opt.x % np.pi
    )

    best_info = tif_fixed_theta_score(
        J,
        theta_best,
    )

    best_rel = float(
        best_info[
            "relative"
        ]
    )

    median_rel = float(
        np.median(
            coarse_rel
        )
    )

    depth = (
        median_rel
        /
        max(
            best_rel,
            1.0e-15,
        )
    )

    return {
        "support": J,

        "size": len(J),

        "best_relative":
            best_rel,

        "best_rms":
            float(
                best_info[
                    "rms"
                ]
            ),

        "theta_rad":
            theta_best,

        "theta_deg":
            np.degrees(
                theta_best
            ),

        "depth":
            depth,

        "median_relative":
            median_rel,

        "rank_at_best":
            int(
                best_info[
                    "rank"
                ]
            ),

        "coarse_profile":
            coarse_rel,
    }


# =============================================================================
# 8. REFERENCE SCORES:
#
#       empty support
#       true support
# =============================================================================

print("\nProfiling reference supports...")
print("-" * 112)


score_empty = tif_profile_support(
    ()
)

score_true = tif_profile_support(
    J_true
)


true_theta_deg = np.degrees(
    np.arctan2(
        oracle_pair_law[1],
        oracle_pair_law[0],
    )
) % 180.0


print(
    f"empty support relative residual : "
    f"{score_empty['best_relative']:.6e}"
)

print(
    f"true support relative residual  : "
    f"{score_true['best_relative']:.6e}"
)

print(
    f"true support RMS residual       : "
    f"{score_true['best_rms']:.6e}"
)

print(
    f"profiled law angle             : "
    f"{score_true['theta_deg']:.6f} deg"
)

print(
    f"oracle law angle               : "
    f"{true_theta_deg:.6f} deg"
)

print(
    f"law-angle error                : "
    f"{abs(score_true['theta_deg'] - true_theta_deg):.6e} deg"
)

print(
    f"true-support profile depth      : "
    f"{score_true['depth']:.6e}"
)


# =============================================================================
# 9. ALL LEAVE-ONE-OUT TRUE SUPPORTS
#
# Test:
#
#       J* \ {q}
#
# for every true transition event q.
#
# If every one of these is clearly above the true-support floor, then
# every true event is individually necessary for the profiled model.
# =============================================================================

print("\nProfiling all leave-one-out supports...")
print("-" * 112)


loo_results = []


for q_remove in J_true:

    J_loo = tuple(
        q
        for q in J_true
        if q != q_remove
    )

    result = tif_profile_support(
        J_loo
    )

    result[
        "removed_event"
    ] = q_remove

    loo_results.append(
        result
    )


loo_results = sorted(
    loo_results,
    key=lambda r:
        r["best_relative"],
)


# =============================================================================
# 10. RANDOM ONE-SWAP SUPPORTS
#
# Harder negative controls:
#
#       remove one true event
#       add one false event
#
# Cardinality remains exactly |J*|.
# =============================================================================

rng = np.random.default_rng(
    20260826
)

N_ONE_SWAP = 60

one_swap_results = []


for _ in range(
    N_ONE_SWAP
):

    q_remove = int(
        rng.choice(
            J_true
        )
    )

    q_add = int(
        rng.choice(
            false_events
        )
    )

    J_swap = set(
        J_true
    )

    J_swap.remove(
        q_remove
    )

    J_swap.add(
        q_add
    )

    result = tif_profile_support(
        tuple(
            sorted(
                J_swap
            )
        )
    )

    result[
        "removed_event"
    ] = q_remove

    result[
        "added_event"
    ] = q_add

    one_swap_results.append(
        result
    )


one_swap_results = sorted(
    one_swap_results,
    key=lambda r:
        r["best_relative"],
)


# =============================================================================
# 11. RANDOM SAME-CARDINALITY WRONG SUPPORTS
#
# These are broad negative controls.
# =============================================================================

N_RANDOM_WRONG = 30

random_wrong_results = []


for _ in range(
    N_RANDOM_WRONG
):

    while True:

        J_rand = tuple(
            sorted(
                rng.choice(
                    n_events,
                    size=len(
                        J_true
                    ),
                    replace=False,
                ).tolist()
            )
        )

        if set(
            J_rand
        ) != J_true_set:

            break

    result = tif_profile_support(
        J_rand
    )

    overlap = len(
        set(
            J_rand
        )
        &
        J_true_set
    )

    result[
        "true_overlap"
    ] = overlap

    random_wrong_results.append(
        result
    )


random_wrong_results = sorted(
    random_wrong_results,
    key=lambda r:
        r["best_relative"],
)


# =============================================================================
# 12. SUMMARY STATISTICS
# =============================================================================

true_floor = score_true[
    "best_relative"
]


loo_rel = np.array(
    [
        r["best_relative"]
        for r in loo_results
    ],
    dtype=float,
)

swap_rel = np.array(
    [
        r["best_relative"]
        for r in one_swap_results
    ],
    dtype=float,
)

rand_rel = np.array(
    [
        r["best_relative"]
        for r in random_wrong_results
    ],
    dtype=float,
)


print("\n" + "=" * 112)
print("SUPPORT-SEPARATION SUMMARY")
print("-" * 112)

print(
    f"true-support residual          : "
    f"{true_floor:.6e}"
)

print(
    f"leave-one-out minimum          : "
    f"{np.min(loo_rel):.6e}"
)

print(
    f"leave-one-out median           : "
    f"{np.median(loo_rel):.6e}"
)

print(
    f"leave-one-out maximum          : "
    f"{np.max(loo_rel):.6e}"
)

print(
    f"min LOO / true ratio           : "
    f"{np.min(loo_rel) / max(true_floor, 1e-15):.6e}"
)

print()

print(
    f"one-swap minimum               : "
    f"{np.min(swap_rel):.6e}"
)

print(
    f"one-swap median                : "
    f"{np.median(swap_rel):.6e}"
)

print(
    f"one-swap minimum / true        : "
    f"{np.min(swap_rel) / max(true_floor, 1e-15):.6e}"
)

print()

print(
    f"random-wrong minimum           : "
    f"{np.min(rand_rel):.6e}"
)

print(
    f"random-wrong median            : "
    f"{np.median(rand_rel):.6e}"
)

print(
    f"random-wrong minimum / true    : "
    f"{np.min(rand_rel) / max(true_floor, 1e-15):.6e}"
)


# =============================================================================
# 13. MOST DANGEROUS LEAVE-ONE-OUT CASES
# =============================================================================

print("\nMost dangerous leave-one-out supports")
print("-" * 112)

print(
    "rank  removed event                          "
    "rel_residual     ratio_to_true   theta_deg   depth"
)


for rank, result in enumerate(
    loo_results[:10],
    start=1,
):

    q = result[
        "removed_event"
    ]

    print(
        f"{rank:>4d}  "
        f"{tif_event_description(q):<37s}  "
        f"{result['best_relative']:.3e}      "
        f"{result['best_relative'] / max(true_floor, 1e-15):.3e}      "
        f"{result['theta_deg']:9.4f}   "
        f"{result['depth']:.3e}"
    )


# =============================================================================
# 14. MOST DANGEROUS ONE-SWAP CASES
# =============================================================================

print("\nMost dangerous one-swap supports")
print("-" * 112)

print(
    "rank  removed                               "
    "added                                 "
    "rel_residual   ratio_true"
)


for rank, result in enumerate(
    one_swap_results[:10],
    start=1,
):

    q_remove = result[
        "removed_event"
    ]

    q_add = result[
        "added_event"
    ]

    print(
        f"{rank:>4d}  "
        f"{tif_event_description(q_remove):<37s}  "
        f"{tif_event_description(q_add):<37s}  "
        f"{result['best_relative']:.3e}    "
        f"{result['best_relative'] / max(true_floor, 1e-15):.3e}"
    )


# =============================================================================
# 15. RANDOM-WRONG CONTROLS
# =============================================================================

print("\nBest random same-cardinality wrong supports")
print("-" * 112)

print(
    "rank  true_overlap  rel_residual   ratio_true   theta_deg"
)


for rank, result in enumerate(
    random_wrong_results[:10],
    start=1,
):

    print(
        f"{rank:>4d}  "
        f"{result['true_overlap']:>12d}  "
        f"{result['best_relative']:.3e}    "
        f"{result['best_relative'] / max(true_floor, 1e-15):.3e}    "
        f"{result['theta_deg']:9.4f}"
    )


# =============================================================================
# 16. SIMPLE SEPARATION COUNTS
#
# Count wrong / incomplete supports that lie within fixed multiples of the
# true-support residual.
# =============================================================================

print("\nNear-floor ambiguity counts")
print("-" * 112)

print(
    "factor   leave-one-out   one-swap   random-wrong"
)


for factor in (
    2.0,
    5.0,
    10.0,
    100.0,
):

    threshold = (
        factor
        * true_floor
    )

    print(
        f"{factor:>6.1f}   "
        f"{np.sum(loo_rel <= threshold):>13d}   "
        f"{np.sum(swap_rel <= threshold):>8d}   "
        f"{np.sum(rand_rel <= threshold):>12d}"
    )


# =============================================================================
# 17. SAVE
# =============================================================================

TIF_SUPPORT_SEPARATION_AUDIT = {
    "true_support":
        J_true,

    "true_score":
        score_true,

    "empty_score":
        score_empty,

    "leave_one_out":
        loo_results,

    "one_swap":
        one_swap_results,

    "random_wrong":
        random_wrong_results,

    "theta_grid":
        theta_grid.copy(),

    "oracle_used_for_diagnosis":
        True,

    "oracle_used_for_search":
        False,
}


print("\n" + "=" * 112)
print("CELL 9 PASSED")
print("=" * 112)

TIF — CELL 9
PROFILED SUPPORT-SEPARATION AUDIT

Projected target
----------------------------------------------------------------------------------------------------------------
scalar rows                    : 5760
||y_perp||_2                  : 4.979676e+00

Candidate-event space
----------------------------------------------------------------------------------------------------------------
possible transition-edge events: 140
true changed events             : 38
false candidate events          : 102

Profiling reference supports...
----------------------------------------------------------------------------------------------------------------
empty support relative residual : 1.000000e+00
true support relative residual  : 4.361924e-07
true support RMS residual       : 2.861989e-08
profiled law angle             : 26.144029 deg
oracle law angle               : 26.565051 deg
law-angle error                : 4.210224e-01 deg
true-support profile depth      : 2.138194e+01

Profiling al

In [11]:
# =============================================================================
# TIF STUDY — CELL 10
# OBSERVATION-FLOOR VS STRUCTURAL-AMBIGUITY AUDIT
#
# Purpose
# -------
# Cell 9 found a very strong global support landscape, but a few
# leave-one-out / one-swap supports lie at essentially the same residual
# floor as the true support.
#
# We now determine whether this is caused by:
#
#   A. finite-dt observation / midpoint-gradient error,
#
# or
#
#   B. genuine structural non-uniqueness along the single trajectory.
#
# Diagnostic idea
# ---------------
# Keep EXACTLY the same:
#
#       trajectory states
#       weak quotient library
#       cumulative transition design
#       baseline projection
#
# but replace the observed secant velocity target by the velocity that the
# TRUE quotient model produces at the same midpoint states.
#
# Thus:
#
#       V_oracle[n] = Bq_mid[n] theta_true(stage[n]).
#
# This removes observation/discretization mismatch while preserving the
# same structural geometry.
#
# Oracle is used ONLY for this diagnostic target.
#
# NO support inference.
# =============================================================================

import numpy as np
from scipy.optimize import minimize_scalar


print("=" * 112)
print("TIF — CELL 10")
print("OBSERVATION-FLOOR VS STRUCTURAL-AMBIGUITY AUDIT")
print("=" * 112)


# =============================================================================
# 0. BUILD ORACLE-CONSISTENT MIDPOINT VELOCITY TARGET
# =============================================================================

V_oracle_consistent = np.zeros_like(
    V,
    dtype=float,
)


for stage, sl in enumerate(
    TIF_OBS["segment_slices"]
):

    V_oracle_consistent[sl] = np.einsum(
        "inq,q->in",
        Bq_mid[sl],
        TIF_ORACLE_THETA[stage],
    )


y_oracle = (
    V_oracle_consistent
    .reshape(-1)
)


# =============================================================================
# 1. PROJECT OUT THE SAME SINGLE GLOBAL BASELINE
# =============================================================================

y_oracle_perp = tif_project_perp(
    y_oracle[:, None],
    U_global,
).ravel()


oracle_y2 = float(
    y_oracle_perp
    @ y_oracle_perp
)

oracle_ynorm = np.sqrt(
    oracle_y2
)

oracle_nrows = len(
    y_oracle_perp
)


# Compare observed and oracle-consistent projected targets.
target_difference = (
    np.linalg.norm(
        y_global_perp
        -
        y_oracle_perp
    )
    /
    np.linalg.norm(
        y_oracle_perp
    )
)


print("\nProjected targets")
print("-" * 112)

print(
    f"||observed y_perp||            : "
    f"{np.linalg.norm(y_global_perp):.6e}"
)

print(
    f"||oracle-consistent y_perp||   : "
    f"{oracle_ynorm:.6e}"
)

print(
    f"relative target mismatch       : "
    f"{target_difference:.6e}"
)


# =============================================================================
# 2. TARGET-SPECIFIC GRAM CROSS TERMS
#
# Structural Gram matrices G11/G12/G22 do NOT change.
# Only S^T y changes.
# =============================================================================

oracle_g1 = (
    S1_perp.T
    @ y_oracle_perp
)

oracle_g2 = (
    S2_perp.T
    @ y_oracle_perp
)


# =============================================================================
# 3. FIXED-THETA SCORE FOR ORACLE-CONSISTENT TARGET
# =============================================================================

def tif_fixed_theta_score_oracle_target(
    J,
    theta,
    eig_rtol=1.0e-12,
):

    J = np.asarray(
        J,
        dtype=int,
    )

    if len(J) == 0:

        return {
            "relative": 1.0,
            "rms": (
                oracle_ynorm
                /
                np.sqrt(
                    oracle_nrows
                )
            ),
            "delta": np.zeros(
                0,
                dtype=float,
            ),
        }

    theta = float(
        theta % np.pi
    )

    c1 = np.cos(
        theta
    )

    c2 = np.sin(
        theta
    )

    G = (
        c1**2
        * G11[
            np.ix_(J, J)
        ]
        +
        c1 * c2
        * (
            G12[
                np.ix_(J, J)
            ]
            +
            G12.T[
                np.ix_(J, J)
            ]
        )
        +
        c2**2
        * G22[
            np.ix_(J, J)
        ]
    )

    G = 0.5 * (
        G + G.T
    )

    g = (
        c1
        * oracle_g1[J]
        +
        c2
        * oracle_g2[J]
    )

    evals, evecs = np.linalg.eigh(
        G
    )

    lam_max = float(
        np.max(
            evals
        )
    )

    if lam_max <= 0.0:

        delta = np.zeros(
            len(J),
            dtype=float,
        )

    else:

        keep = (
            evals
            >
            eig_rtol
            * lam_max
        )

        if not np.any(
            keep
        ):

            delta = np.zeros(
                len(J),
                dtype=float,
            )

        else:

            Vkeep = evecs[
                :,
                keep
            ]

            lam = evals[
                keep
            ]

            delta = (
                Vkeep
                @ (
                    (
                        Vkeep.T @ g
                    )
                    /
                    lam
                )
            )

    # ---------------------------------------------------------------------
    # IMPORTANT:
    #
    # Compute the FINAL residual directly in trajectory-row space.
    # This avoids catastrophic cancellation of
    #
    #       ||y||^2 - g^T G^+ g
    #
    # when the true residual approaches machine precision.
    # ---------------------------------------------------------------------

    S_theta = (
        c1
        * S1_perp[:, J]
        +
        c2
        * S2_perp[:, J]
    )

    residual = (
        y_oracle_perp
        -
        S_theta @ delta
    )

    residual_norm = np.linalg.norm(
        residual
    )

    return {
        "relative": (
            residual_norm
            /
            oracle_ynorm
        ),

        "rms": (
            residual_norm
            /
            np.sqrt(
                oracle_nrows
            )
        ),

        "delta": delta,
    }


# =============================================================================
# 4. PROFILE SHARED-LAW ANGLE
#
# Use the same deterministic coarse grid as Cell 9, then refine.
# =============================================================================

def tif_profile_support_oracle_target(
    J,
):

    J = tuple(
        sorted(
            set(
                int(q)
                for q in J
            )
        )
    )

    coarse = np.empty(
        len(theta_grid),
        dtype=float,
    )

    for i, theta in enumerate(
        theta_grid
    ):

        coarse[i] = (
            tif_fixed_theta_score_oracle_target(
                J,
                theta,
            )["relative"]
        )

    ibest = int(
        np.argmin(
            coarse
        )
    )

    theta0 = float(
        theta_grid[
            ibest
        ]
    )

    dtheta = (
        np.pi
        /
        len(
            theta_grid
        )
    )

    def objective(theta):

        return (
            tif_fixed_theta_score_oracle_target(
                J,
                theta % np.pi,
            )["relative"]
        )

    opt = minimize_scalar(
        objective,
        bounds=(
            theta0 - dtheta,
            theta0 + dtheta,
        ),
        method="bounded",
        options={
            "xatol": 1.0e-12,
            "maxiter": 120,
        },
    )

    theta_best = float(
        opt.x % np.pi
    )

    final = (
        tif_fixed_theta_score_oracle_target(
            J,
            theta_best,
        )
    )

    return {
        "support":
            J,

        "size":
            len(J),

        "best_relative":
            float(
                final[
                    "relative"
                ]
            ),

        "best_rms":
            float(
                final[
                    "rms"
                ]
            ),

        "theta_deg":
            float(
                np.degrees(
                    theta_best
                )
            ),

        "median_relative":
            float(
                np.median(
                    coarse
                )
            ),
    }


# =============================================================================
# 5. TRUE SUPPORT
# =============================================================================

print("\nProfiling true support on oracle-consistent target...")
print("-" * 112)


oracle_score_true = (
    tif_profile_support_oracle_target(
        J_true
    )
)


print(
    f"true-support relative residual : "
    f"{oracle_score_true['best_relative']:.6e}"
)

print(
    f"true-support RMS residual      : "
    f"{oracle_score_true['best_rms']:.6e}"
)

print(
    f"profiled law angle            : "
    f"{oracle_score_true['theta_deg']:.9f} deg"
)

print(
    f"oracle law angle               : "
    f"{true_theta_deg:.9f} deg"
)

print(
    f"angle error                    : "
    f"{abs(oracle_score_true['theta_deg'] - true_theta_deg):.6e} deg"
)


# =============================================================================
# 6. ALL LEAVE-ONE-OUT SUPPORTS
# =============================================================================

print("\nProfiling all leave-one-out supports...")
print("-" * 112)


oracle_loo_results = []


for q_remove in J_true:

    J_loo = tuple(
        q
        for q in J_true
        if q != q_remove
    )

    result = (
        tif_profile_support_oracle_target(
            J_loo
        )
    )

    result[
        "removed_event"
    ] = q_remove

    oracle_loo_results.append(
        result
    )


oracle_loo_results = sorted(
    oracle_loo_results,
    key=lambda r:
        r[
            "best_relative"
        ],
)


# =============================================================================
# 7. RE-EVALUATE THE EXACT SAME ONE-SWAP CONTROLS AS CELL 9
#
# This is important:
# do NOT draw a new random sample.
#
# We want a direct before/after comparison for the same wrong supports.
# =============================================================================

oracle_swap_results = []


seen_swap_supports = set()


for old_result in one_swap_results:

    q_remove = int(
        old_result[
            "removed_event"
        ]
    )

    q_add = int(
        old_result[
            "added_event"
        ]
    )

    J_swap = set(
        J_true
    )

    J_swap.remove(
        q_remove
    )

    J_swap.add(
        q_add
    )

    J_swap = tuple(
        sorted(
            J_swap
        )
    )

    if J_swap in seen_swap_supports:
        continue

    seen_swap_supports.add(
        J_swap
    )

    result = (
        tif_profile_support_oracle_target(
            J_swap
        )
    )

    result[
        "removed_event"
    ] = q_remove

    result[
        "added_event"
    ] = q_add

    oracle_swap_results.append(
        result
    )


oracle_swap_results = sorted(
    oracle_swap_results,
    key=lambda r:
        r[
            "best_relative"
        ],
)


# =============================================================================
# 8. SUMMARY
# =============================================================================

oracle_true_floor = (
    oracle_score_true[
        "best_relative"
    ]
)

oracle_loo_rel = np.array(
    [
        r[
            "best_relative"
        ]
        for r in oracle_loo_results
    ],
    dtype=float,
)

oracle_swap_rel = np.array(
    [
        r[
            "best_relative"
        ]
        for r in oracle_swap_results
    ],
    dtype=float,
)


print("\n" + "=" * 112)
print("FLOOR-VS-STRUCTURAL-AMBIGUITY SUMMARY")
print("-" * 112)

print(
    f"observed true-support residual : "
    f"{score_true['best_relative']:.6e}"
)

print(
    f"oracle true-support residual   : "
    f"{oracle_true_floor:.6e}"
)

print()

print(
    f"oracle LOO minimum             : "
    f"{np.min(oracle_loo_rel):.6e}"
)

print(
    f"oracle LOO median              : "
    f"{np.median(oracle_loo_rel):.6e}"
)

print()

print(
    f"oracle one-swap minimum        : "
    f"{np.min(oracle_swap_rel):.6e}"
)

print(
    f"oracle one-swap median         : "
    f"{np.median(oracle_swap_rel):.6e}"
)


# =============================================================================
# 9. MOST DANGEROUS ORACLE-CONSISTENT LOO CASES
# =============================================================================

print("\nMost dangerous oracle-consistent leave-one-out supports")
print("-" * 112)

print(
    "rank  removed event                          "
    "oracle_rel      observed_rel"
)


observed_loo_by_removed = {
    int(
        r[
            "removed_event"
        ]
    ):
    r[
        "best_relative"
    ]
    for r in loo_results
}


for rank, result in enumerate(
    oracle_loo_results[
        :10
    ],
    start=1,
):

    q = int(
        result[
            "removed_event"
        ]
    )

    print(
        f"{rank:>4d}  "
        f"{tif_event_description(q):<37s}  "
        f"{result['best_relative']:.3e}    "
        f"{observed_loo_by_removed[q]:.3e}"
    )


# =============================================================================
# 10. MOST DANGEROUS ORACLE-CONSISTENT ONE-SWAPS
# =============================================================================

print("\nMost dangerous oracle-consistent one-swap supports")
print("-" * 112)

print(
    "rank  removed                               "
    "added                                 "
    "oracle_rel"
)


for rank, result in enumerate(
    oracle_swap_results[
        :10
    ],
    start=1,
):

    print(
        f"{rank:>4d}  "
        f"{tif_event_description(result['removed_event']):<37s}  "
        f"{tif_event_description(result['added_event']):<37s}  "
        f"{result['best_relative']:.3e}"
    )


# =============================================================================
# 11. SPECIFICALLY TRACK CELL-9'S WORST AMBIGUITY
# =============================================================================

danger_remove = tif_event_id(
    0,
    pair_index[
        (0, 1)
    ],
)

danger_candidates = [
    r
    for r in oracle_loo_results
    if (
        r[
            "removed_event"
        ]
        ==
        danger_remove
    )
]


print("\nCell-9 weakest true event")
print("-" * 112)

print(
    "event                         : "
    f"{tif_event_description(danger_remove)}"
)

print(
    "observed leave-one-out score  : "
    f"{observed_loo_by_removed[danger_remove]:.6e}"
)

print(
    "oracle-consistent LOO score   : "
    f"{danger_candidates[0]['best_relative']:.6e}"
)


# =============================================================================
# 12. SAVE
# =============================================================================

TIF_FLOOR_AMBIGUITY_AUDIT = {
    "oracle_target":
        y_oracle_perp,

    "target_relative_mismatch":
        target_difference,

    "true_score":
        oracle_score_true,

    "leave_one_out":
        oracle_loo_results,

    "one_swap":
        oracle_swap_results,

    "oracle_used_for_diagnosis":
        True,

    "oracle_used_for_inference":
        False,
}


print("\n" + "=" * 112)
print("CELL 10 PASSED")
print("=" * 112)

TIF — CELL 10
OBSERVATION-FLOOR VS STRUCTURAL-AMBIGUITY AUDIT

Projected targets
----------------------------------------------------------------------------------------------------------------
||observed y_perp||            : 4.979676e+00
||oracle-consistent y_perp||   : 4.979677e+00
relative target mismatch       : 2.698072e-07

Profiling true support on oracle-consistent target...
----------------------------------------------------------------------------------------------------------------
true-support relative residual : 4.374189e-07
true-support RMS residual      : 2.870037e-08
profiled law angle            : 26.150174855 deg
oracle law angle               : 26.565051177 deg
angle error                    : 4.148763e-01 deg

Profiling all leave-one-out supports...
----------------------------------------------------------------------------------------------------------------

FLOOR-VS-STRUCTURAL-AMBIGUITY SUMMARY
------------------------------------------------------------------